## 1. Import Constants and Units

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
# Plot style shared by all figures in this notebook.
font = {"size": 14, "family": "STIXGeneral"}
mpl.rc('font', **font)
mpl.rc('axes', labelsize=16) 
plt.rcParams['legend.fontsize']=14
plt.rcParams["figure.figsize"] = [6.0,4.]
# plt.rcParams["xtick.labelsize"] = 16; plt.rcParams["ytick.labelsize"] = 16
mpl.rc('text', usetex=True)

plt.rcParams["figure.figsize"] = (5, 3)
plt.rcParams["axes.grid"]=True;plt.rcParams["grid.alpha"]=0.4; plt.rcParams["grid.color"]='#999999'; plt.rcParams["grid.linestyle"]='--'

In [ ]:
DATA_DIR  = "/Users/chase/glxy_mass/MTNG_data"     # <-- edit
# SNAPSHOTS is the ONE control for redshift scope. List both to POOL them (the
# default, main analysis); list a single snapshot to run that epoch on its own
# (e.g. SNAPSHOTS = ["z=0.5"]). Everything downstream -- loading, the per-halo
# scale factor a = 1/(1+z), the E(z) normalization, the split, and the weights --
# is driven off this list, so changing it and re-running from the Load cell (sec. 5)
# is all that is needed. Per-halo z / E(z) / a are attached in that Load cell.
SNAPSHOTS = ["z=0.0", "z=0.5"]
PLOTDIR   = "figs/"
os.makedirs(PLOTDIR, exist_ok=True)

In [ ]:
# ---- cosmology -------------------------------------------------------------
H_LITTLE = 0.6774
OMEGA_M  = 0.3089
OMEGA_LAMBDA = 1.0 - OMEGA_M          # flat LCDM
XH       = 0.76 

def Ez(z):
    """Dimensionless Hubble rate E(z) = H(z)/H0 for flat LCDM. Vectorized."""
    return np.sqrt(OMEGA_M * (1.0 + z) ** 3 + OMEGA_LAMBDA)

# ---- physical constants (CGS) ---------------------------------------------
G_CGS = 6.67430e-8 # in CGS units
PROTON_TO_G = 1.6726219e-24
MSUN_G = 1.98892e33
KEV_ERG   = 1.602176634e-9
KB_KEV_PER_K = 8.617333262e-8    # Boltzmann const [keV/K]: T500c[K] -> kT[keV] for Y_X

MU_E = 2.0 / (1.0 + XH)               # per free electron   ~1.136
MU   = 4.0 / (3.0 + 5.0 * XH)         # per particle        ~0.588
PTH_OVER_PE = MU_E / MU               # P_thermal / P_e     ~1.932


# ---- simulation units -> CGS ----------------------------------------------
TNG_TO_MSUN = 1e10 / H_LITTLE
TNG_TO_G    = TNG_TO_MSUN * MSUN_G
TNG_TO_CM   = 3.0857e24 / H_LITTLE    # Mpc/h -> cm
TNG_TO_S    = 3.15576e16 / H_LITTLE   # Gyr/h -> s


PROTON_TO_TNG = PROTON_TO_G / TNG_TO_G
RHO_SCALING   = MU_E * PROTON_TO_TNG          # n_e profile -> gas mass density
P_SCALING     = TNG_TO_G / TNG_TO_S**2 / TNG_TO_CM * 1e-6   # y profile -> barye
Y_SCALING = 1e-6 * 6.65e-25 / (9.1e-28 * 3e10**2) * 1.98e43 / H_LITTLE / (3.154e16)**2

---
## 2. Radial Grid and Profile Tools

In [ ]:
# Leander's code: https://github.com/leanderthiele/group_particles/tree/master/examples
N_BINS = 128
RBIN_EDGES=np.append(0,np.geomspace(0.03, 2.5, num=128)) # radial bins in Leander's code
RBIN_OUTER = RBIN_EDGES[1:]                       # outer edge of each bin
VOL_BIN    = (4/3) * np.pi * (RBIN_EDGES[1:]**3 - RBIN_EDGES[:-1]**3)

_r1, _r2 = RBIN_EDGES[:-1], RBIN_EDGES[1:]
R_EFF = 0.75 * (_r2**4 - _r1**4) / (_r2**3 - _r1**3)   # volume-weighted centres replaced old r_bins

# gNFW shape parameters held fixed; only (lnP0, ln xc, beta) are free
GNFW_ALPHA, GNFW_GAMMA = 1.0, -0.3
FIT_RANGE   = (0.10, 1.20)          # swept in section 14
SLOPE_WINDOW = (0.70, 1.40)         # for the model-free local slope
MASS_CUT     = 1e4                  # 1e10 Msun/h  -> 1e14 Msun/h
MAX_LOG_RESID = 0.25                # gNFW fit-quality cut

print(f"bin 0 spans [0, {RBIN_EDGES[1]:.3f}] R200c (a sphere, excluded everywhere)")
print(f"old 'R200c' index 101 of a cumulative profile = "
      f"{RBIN_OUTER[101]:.4f} R200c   <-- the 1.1% aperture error")

In [ ]:
def cumulative_profile(density_profiles, r200, vol_bin=VOL_BIN):
    '''Integrate shell densities into enclosed totals. Pure: never mutates input.'''
    prof = np.asarray(density_profiles, dtype=float)
    if prof.ndim == 1:
        prof = prof[None, :]
        squeeze = True
    else:
        squeeze = False
    r200 = np.atleast_1d(np.asarray(r200, dtype=float))
    shells = prof * (r200[:, None]**3) * vol_bin[None, :]
    out = np.cumsum(shells, axis=1)
    return out[0] if squeeze else out


def aperture_value(cumulative, r_target, r_outer=RBIN_OUTER):
    '''Enclosed quantity inside r_target (in r/R200c), log-log interpolated.'''
    cum = np.atleast_2d(np.asarray(cumulative, dtype=float))
    ln_r, ln_t = np.log(r_outer), np.log(r_target)
    out = np.full(cum.shape[0], np.nan)
    for i, row in enumerate(cum):
        g = np.isfinite(row) & (row > 0)
        if g.sum() >= 2:
            out[i] = np.exp(np.interp(ln_t, ln_r[g], np.log(row[g])))
    return out if np.ndim(cumulative) == 2 else out[0]


def interp_profile(profile, r_target, r_eff=R_EFF):
    '''Log-log interpolate a density-like profile to r_target.'''
    prof = np.asarray(profile, dtype=float)
    g = np.isfinite(prof) & (prof > 0)
    if g.sum() < 2:
        return np.nan
    return float(np.exp(np.interp(np.log(r_target), np.log(r_eff[g]), np.log(prof[g]))))


def concentration(cumulative, r_inner, r_outer_ap=1.0):
    '''Aperture concentration Q(<r_inner) / Q(<r_outer_ap).'''
    with np.errstate(divide="ignore", invalid="ignore"):
        return aperture_value(cumulative, r_inner) / aperture_value(cumulative, r_outer_ap)

---
## 3. gNFW Pressure

In [ ]:
def gnfw_lnP(ln_x, ln_P0, ln_xc, beta, alpha=GNFW_ALPHA, gamma=GNFW_GAMMA):
    ln_u  = ln_x - ln_xc
    a_lnu = alpha * ln_u
    ln_1pu = np.where(a_lnu > 30.0, a_lnu, np.log1p(np.exp(np.minimum(a_lnu, 30.0))))
    return ln_P0 + gamma * ln_u - beta * ln_1pu


def gnfw_logslope(ln_x, ln_xc, beta, alpha=GNFW_ALPHA, gamma=GNFW_GAMMA):
    '''Analytic dlnP/dlnr. Negative for beta > 0.'''
    a_lnu = alpha * (np.asarray(ln_x, dtype=float) - ln_xc)
    frac  = 1.0 / (1.0 + np.exp(-np.clip(a_lnu, -700, 700)))
    return gamma - alpha * beta * frac


def fit_gnfw(r, pressure, fit_range=FIT_RANGE):
    '''Fit one profile. Returns a dict (nan-filled and converged=False on failure).'''
    r, p = np.asarray(r, float), np.asarray(pressure, float)
    ok = np.isfinite(p) & (p > 0) & (r >= fit_range[0]) & (r <= fit_range[1])
    bad = dict(ln_P0=np.nan, ln_xc=np.nan, beta=np.nan, cov=np.full((3, 3), np.nan),
               rms=np.nan, converged=False, n=int(ok.sum()))
    if ok.sum() < 5:
        return bad

    ln_x, ln_P = np.log(r[ok]), np.log(p[ok])
    slope0 = (ln_P[-1] - ln_P[0]) / (ln_x[-1] - ln_x[0])
    beta0  = float(np.clip(-(slope0 - GNFW_GAMMA), 0.5, 20.0))
    lnxc0  = np.log(0.5)
    mid    = len(ln_x) // 2
    lnP00  = float(ln_P[mid] - gnfw_lnP(ln_x[mid], 0.0, lnxc0, beta0))

    try:
        popt, pcov = curve_fit(gnfw_lnP, ln_x, ln_P, p0=(lnP00, lnxc0, beta0),
                               maxfev=20000)
    except (RuntimeError, ValueError):
        return bad

    resid = ln_P - gnfw_lnP(ln_x, *popt)
    return dict(ln_P0=float(popt[0]), ln_xc=float(popt[1]), beta=float(popt[2]),
                cov=np.asarray(pcov, float),
                rms=float(np.sqrt(np.mean(resid**2))),
                converged=bool(np.all(np.isfinite(popt)) and popt[2] > 0),
                n=int(ok.sum()))


def slope_sigma(fit, r=1.0):
    '''1-sigma on the log-slope, propagated from the fit covariance.

    This is why beta and x_c should not be used as features directly: they are
    strongly anti-correlated, but their combination -- the slope -- is well
    determined. sigma(slope)/|slope| is typically 4x smaller than sigma(b)/beta.
    '''
    if not np.all(np.isfinite(fit["cov"])):
        return np.nan
    a = GNFW_ALPHA
    u_a = np.exp(a * (np.log(r) - fit["ln_xc"]))
    f = u_a / (1.0 + u_a)
    jac = np.array([0.0, a*a*fit["beta"]*f*(1-f), -a*f])
    v = float(jac @ fit["cov"] @ jac)
    return float(np.sqrt(v)) if v > 0 else np.nan


def local_log_slope(r, pressure, r_target=1.0, window=SLOPE_WINDOW):
    '''Model-free dlnP/dlnr from a local power-law fit. Independent of gNFW.'''
    r, p = np.asarray(r, float), np.asarray(pressure, float)
    ok = np.isfinite(p) & (p > 0) & (r >= window[0]) & (r <= window[1])
    if ok.sum() < 3:
        return np.nan
    return float(np.polyfit(np.log(r[ok] / r_target), np.log(p[ok]), 1)[0])


---
## 4. Hydrostatic mass and bias

$$M_{\rm HSE}(r)=-\frac{r\,P_{\rm th}(r)}{G\,\rho_{\rm gas}(r)}\,
\frac{d\ln P}{d\ln r}\bigg|_r,
\qquad b\equiv 1-\frac{M_{\rm HSE}}{M_{200c}}$$

In [ ]:
def hse_bias(P_e_prof, rho_prof, r200, m200_tng, r_eval=1.0,
             fit=None, fit_range=FIT_RANGE, slope_source="fit", p_source="data",
             a_scale=1.0):
    '''Everything for one halo at r_eval. Returns a dict.'''
    out = dict(m_hse=np.nan, bias=np.nan, slope_fit=np.nan, slope_local=np.nan,
               slope_sig=np.nan, kT_keV=np.nan, P_th=np.nan, rho=np.nan,
               rms=np.nan, beta=np.nan, xc=np.nan, ok=False)

    if fit is None:
        fit = fit_gnfw(R_EFF, P_e_prof, fit_range=fit_range)
    out["rms"], out["beta"] = fit["rms"], fit["beta"]
    out["xc"] = np.exp(fit["ln_xc"]) if np.isfinite(fit["ln_xc"]) else np.nan
    if not fit["converged"]:
        return out

    out["slope_fit"]   = float(gnfw_logslope(np.log(r_eval), fit["ln_xc"], fit["beta"]))
    out["slope_fit_0p65"]   = float(gnfw_logslope(np.log(r_eval*0.65), fit["ln_xc"], fit["beta"]))
    out["slope_sig"]   = slope_sigma(fit, r_eval)
    out["slope_local"] = local_log_slope(R_EFF, P_e_prof, r_target=r_eval)
    slope = out["slope_fit"] if slope_source == "fit" else out["slope_local"]

    if p_source == "data":
        P_e_at_r = interp_profile(P_e_prof, r_eval)
    else:
        P_e_at_r = float(np.exp(gnfw_lnP(np.log(r_eval), fit["ln_P0"],
                                         fit["ln_xc"], fit["beta"])))
    rho_at_r = interp_profile(rho_prof, r_eval)
    if not (np.isfinite(P_e_at_r) and np.isfinite(rho_at_r) and rho_at_r > 0):
        return out

    out["P_th"] = P_e_at_r * PTH_OVER_PE
    out["rho"]  = rho_at_r
    r_cm = r_eval * r200 * a_scale * TNG_TO_CM   # physical radius = comoving * a (a=1 at z=0)
    m_hse_g = -(r_cm * out["P_th"] / (G_CGS * rho_at_r)) * slope

    out["m_hse"] = m_hse_g / MSUN_G
    out["bias"]  = 1.0 - out["m_hse"] / (m200_tng * TNG_TO_MSUN)
    out["kT_keV"] = MU * PROTON_TO_G * out["P_th"] / rho_at_r / KEV_ERG   # unit canary
    out["ok"] = bool(np.isfinite(out["bias"]) and out["m_hse"] > 0
                     and fit["rms"] < MAX_LOG_RESID)
    return out

---
## 5. Load

In [ ]:
# Pool the requested snapshots into one set of arrays. Per-halo z / E(z) / a
# arrays are carried alongside so the HSE physical radius (= comoving * a) and
# the E(z) normalization downstream are applied per halo.
def _load_snap_arrays(snap):
    d = f"{DATA_DIR}/{snap}"
    names = dict(Y="Y200c", m200="M200c", r200="R200c", T500c="T500c",
                 yProf="y_profiles", neProf="ne_profiles", mStarProf="mStar_profiles")
    return {k: np.load(f"{d}/{fn}.npy") for k, fn in names.items()}

_parts = [(_snap, _load_snap_arrays(_snap)) for _snap in SNAPSHOTS]
def _cat(key):
    return np.concatenate([p[key] for _s, p in _parts], axis=0)

Y         = _cat("Y")
m200      = _cat("m200")
r200      = _cat("r200")
T500c_all = _cat("T500c")                                      # spectroscopic-like kT proxy at R500c [K], for Y_X
yProf     = _cat("yProf")
neProf    = _cat("neProf")
mStarProf = _cat("mStarProf")

z_all       = np.concatenate([np.full(len(p["m200"]), float(_s.split("=")[1])) for _s, p in _parts])
Ez_all      = Ez(z_all)                                        # E(z) per halo
a_scale_all = 1.0 / (1.0 + z_all)                              # comoving -> physical length factor per halo

P_e_all = yProf  * P_SCALING                                   # barye
rho_all = neProf * RHO_SCALING * TNG_TO_G / TNG_TO_CM**3       # g/cm^3

print("pooled " + ", ".join(f"{_s}={len(p['m200'])}" for _s, p in _parts) + f"  ->  {len(m200)} halos")

In [ ]:
print(f"{len(m200)} halos pooled from {SNAPSHOTS}")
print(f"M200c: {m200.min()*TNG_TO_MSUN:.2e} - {m200.max()*TNG_TO_MSUN:.2e} Msun")
print(f"profiles: {yProf.shape}")

sel = np.where(m200 > MASS_CUT)[0]
print(f"\n{len(sel)} halos above M200c > {MASS_CUT*1e10:.0e} Msun/h")

### Unit Check

In [ ]:
rng   = np.random.default_rng(0)
probe = rng.choice(sel, size=min(200, len(sel)), replace=False)
rows  = [hse_bias(P_e_all[i], rho_all[i], r200[i], m200[i], a_scale=a_scale_all[i]) for i in probe]
good  = [r for r in rows if r["ok"]]

RANGES = {"kT_keV": (0.5, 12.0), "bias": (-0.15, 0.55), "slope_fit": (-6.0, -1.5)}
for k, (lo, hi) in RANGES.items():
    v = np.array([r[k] for r in good])
    frac = np.mean((v > lo) & (v < hi))
    print(f"{'OK ' if frac > 0.9 else '!! '}{k:<10} median {np.median(v):+8.3f}  "
          f"16-84 [{np.percentile(v,16):+.3f}, {np.percentile(v,84):+.3f}]  "
          f"{frac:.0%} inside {lo, hi}")

cg = cumulative_profile(neProf[probe] * RHO_SCALING, r200[probe])
fg = aperture_value(cg, 1.0) / m200[probe]
print(f"OK  f_gas      median {np.median(fg):.4f}  "
      f"16-84 [{np.percentile(fg,16):.4f}, {np.percentile(fg,84):.4f}]")

---
## 6. Build Halo Table

In [ ]:
cumY   = cumulative_profile(yProf[sel] * Y_SCALING, r200[sel])
cumGas = cumulative_profile(neProf[sel] * RHO_SCALING, r200[sel])
cumStr = cumulative_profile(mStarProf[sel], r200[sel])

mGas  = aperture_value(cumGas, 1.0)
mStar = aperture_value(cumStr, 1.0)
mGas500 = aperture_value(cumGas, 0.65)              # gas mass in R500 (~0.65 R200) aperture, for Y_X

# --- core-excised, gas-mass-weighted ICM temperature at R500c ---------------
# NOTE: T500c.npy is a VOLUME-weighted mean over the FULL 0->R500 aperture
# (Leander's meanT.cpp) -- core-INCLUDED, NOT the observers' core-excised X-ray
# temperature. Rebuild a core-excised, gas-mass-weighted kT from the pressure &
# density profiles: T(r) = mu*m_p*P_th/rho (identical to hse_bias's kT_keV),
# averaged over the standard X-ray annulus [0.15, 1]*R500 (R500 ~ 0.65 R200).
R500_OVER_R200 = 0.65                       # R500c ~ 0.65 R200c proxy (matches mGas500)
CORE_FRAC      = 0.15                        # inner core excised: annulus = [0.15, 1]*R500
_annulus = (R_EFF >= CORE_FRAC * R500_OVER_R200) & (R_EFF <= R500_OVER_R200)   # 54 of 128 bins
with np.errstate(divide="ignore", invalid="ignore"):
    _kT_bin = MU * PROTON_TO_G * (P_e_all[sel] * PTH_OVER_PE) / rho_all[sel] / KEV_ERG  # keV per bin
    _w_mass = rho_all[sel] * VOL_BIN[None, :]          # gas-mass weight (for VOLUME-wt use VOL_BIN alone)
_w_mass = np.where(_annulus[None, :] & np.isfinite(_kT_bin) & np.isfinite(_w_mass) & (_w_mass > 0),
                   _w_mass, 0.0)
_kT_bin = np.where(np.isfinite(_kT_bin), _kT_bin, 0.0)
_wsum   = _w_mass.sum(axis=1)
kT500c_ce = np.where(_wsum > 0, (_w_mass * _kT_bin).sum(axis=1) / _wsum, np.nan)   # keV, core-excised, mass-wt


T = {
    "halo_id":     sel.astype(float),
    "z":           z_all[sel],
    "E_z":         Ez_all[sel],
    "a_scale":     a_scale_all[sel],
    "M200c":       m200[sel] * TNG_TO_MSUN,          # DIAGNOSTIC ONLY
    "R200c":       r200[sel],
    "M_gas":       mGas  * TNG_TO_MSUN,
    "M_star":      mStar * TNG_TO_MSUN,
    "Mgas500":     mGas500 * TNG_TO_MSUN,                 # X-ray gas mass within R500 [Msun]
    "kT500c":      kT500c_ce,                            # core-excised gas-mass-weighted kT, [0.15,1]*R500 [keV] (was: stored vol-wt full-aperture T500c)
    "Y_X_500":     mGas500 * TNG_TO_MSUN * kT500c_ce,    # Kravtsov Y_X = Mgas500 * core-excised kT500c [Msun keV]; Mgas500 still uses 0.65 R200 proxy
    "Y200_HSE":    aperture_value(cumY, 1.0),
    "Y500_HSE":    aperture_value(cumY, 0.65),
    "c_Y_0p50":    concentration(cumY,   0.50),
    "c_Y_0p15":    concentration(cumY,   0.15),
    "c_Y_0p25":    concentration(cumY,   0.25),
    "c_gas_0p50":  concentration(cumGas, 0.50),
    "c_gas_0p15":  concentration(cumGas, 0.15),
    "Y_ratio_0p65":concentration(cumY,   0.65),
    "Mstar_over_Mgas": mStar / mGas,
    "f_gas_true":  mGas / m200[sel],                 # DIAGNOSTIC ONLY
}
concentration_keys = [c for c in ["c_Y_0p50", "c_Y_0p15", "c_Y_0p25", "c_gas_0p50", "c_gas_0p15"] if c in T]
for k in concentration_keys:
    c = np.clip(T[k], 1e-6, 1 - 1e-6)
    T[f"logit_{k}"] = np.log(c / (1 - c))

hse_keys = ["m_hse", "bias", "slope_fit", "slope_fit_0p65", "slope_local", "slope_sig",
            "kT_keV", "rms", "beta", "xc"]
for k in hse_keys:
    T[k] = np.full(len(sel), np.nan)
T["ok"] = np.zeros(len(sel), bool)

for j, i in enumerate(sel):
    if j % 500 == 0:
        print(f"  {j}/{len(sel)}", flush=True)
    r = hse_bias(P_e_all[i], rho_all[i], r200[i], m200[i], a_scale=a_scale_all[i])
    for k in hse_keys:
        T[k][j] = r[k]
    T["ok"][j] = r["ok"]

T["M_hse"]     = T["m_hse"]
T["f_gas_hse"] = T["M_gas"] / T["M_hse"]     # DIAGNOSTIC, since reparametrizes the target 
T["gnfw_rms_resid"] = T["rms"]
T["slope_R200"]     = T["slope_fit"]
T["slope_R200_0p65"]     = T["slope_fit_0p65"]
T["kT_R200"]        = T["kT_keV"]
T["slope_local_R200"] = T["slope_local"]
T["Y_X_200"] = T["M_gas"] * T["kT_R200"]                 # R200-aperture X-ray analog: Mgas(<R200)*kT_R200 [Msun keV]

qual = T["ok"] & (T["gnfw_rms_resid"] < MAX_LOG_RESID)
print(f"\n{T['ok'].sum()} converged / {len(sel)}   ({T['ok'].mean():.1%})")
print(f"{qual.sum()} also pass the fit-quality cut")
print(f"\nb: median {np.median(T['bias'][qual]):+.4f}, "
      f"mean {np.mean(T['bias'][qual]):+.4f}, scatter {np.std(T['bias'][qual]):.4f}")

---
## 7. Gas morphology parameters

The file holds a flattened symmetric 3×3 mass-weighted second-moment tensor per
halo (verified symmetric to 3e-7, positive definite for all 37371 z=0 halos).
Semi-axes are `sqrt(eigenvalues)`; median minor/major is 0.90, the expected
roundness for cluster gas.

**The 3-D axis ratios are not observable.** What a telescope measures is the
*projected* isophotal ellipticity. The projection is exact for second moments —
integrating out the line of sight leaves the in-sky 2×2 block of `T` untouched,
since $\int xy\,\rho\,dz\,dx\,dy=\int xy\,\Sigma\,dx\,dy$ — so we rotate `T` so
the line of sight lies along one axis and take the remaining 2×2 block.

Validation below: an oblate spheroid with `c/a = 0.5` must give `e = 0` face-on
and `e = 0.5` edge-on.

One random orientation per halo is the *realistic* choice — projection scatter
is part of the measurement, not noise to average away. The orientation-averaged
version is computed too, so you can ask how much of any shape signal is being
diluted by viewing angle.

In [ ]:
def load_gas_tensor(path):
    a = np.load(path).astype(float)
    if a.ndim == 2 and a.shape[1] == 9:
        a = a.reshape(-1, 3, 3)
    return 0.5 * (a + np.transpose(a, (0, 2, 1)))          # symmetrise


def axis_ratios_3d(Tn):
    '''(intermediate/major, minor/major) semi-axis ratios. DIAGNOSTIC ONLY.'''
    w  = np.linalg.eigvalsh(Tn)                            # ascending
    ax = np.sqrt(np.clip(w, 0, None))
    return ax[:, 1] / ax[:, 2], ax[:, 0] / ax[:, 2]


def triaxiality(Tn):
    '''T = (a^2-b^2)/(a^2-c^2): 0 oblate, 1 prolate. DIAGNOSTIC ONLY.'''
    w = np.linalg.eigvalsh(Tn)
    return (w[:, 2] - w[:, 1]) / (w[:, 2] - w[:, 0])


def _sky_basis(n_hat):
    seed = np.zeros_like(n_hat)
    seed[np.arange(len(n_hat)), np.argmin(np.abs(n_hat), axis=1)] = 1.0
    e1 = seed - np.sum(seed * n_hat, axis=1, keepdims=True) * n_hat
    e1 /= np.linalg.norm(e1, axis=1, keepdims=True)
    e2 = np.cross(n_hat, e1)
    e2 /= np.linalg.norm(e2, axis=1, keepdims=True)
    return np.stack([e1, e2], axis=2)


def ellipticity_projected(Tn, seed=0, n_hat=None):
    '''e = 1 - b_2D/a_2D along a random line of sight. THE OBSERVABLE.'''
    Tn = np.asarray(Tn, float)
    if n_hat is None:
        g = np.random.default_rng(seed)
        n_hat = g.normal(size=(len(Tn), 3))
        n_hat /= np.linalg.norm(n_hat, axis=1, keepdims=True)
    B  = _sky_basis(np.asarray(n_hat, float))
    T2 = np.einsum("nij,nia,njb->nab", Tn, B, B)
    w  = np.linalg.eigvalsh(T2)
    with np.errstate(divide="ignore", invalid="ignore"):
        return 1.0 - np.sqrt(np.clip(w[:, 0], 0, None) / np.clip(w[:, 1], 1e-300, None))


def ellipticity_projected_mean(Tn, n_draws=24, seed=0):
    g = np.random.default_rng(seed)
    acc = np.zeros(len(Tn))
    for _ in range(n_draws):
        n = g.normal(size=(len(Tn), 3)); n /= np.linalg.norm(n, axis=1, keepdims=True)
        acc += ellipticity_projected(Tn, n_hat=n)
    return acc / n_draws


# --- validation against analytic spheroids ---------------------------------
_ob = np.diag([1.0, 1.0, 0.25])[None]                      # c/a = 0.5
print(f"oblate c/a=0.5:  face-on e = "
      f"{ellipticity_projected(_ob, n_hat=np.array([[0,0,1.]]))[0]:.4f} (expect 0.0000)")
print(f"                 edge-on e = "
      f"{ellipticity_projected(_ob, n_hat=np.array([[1.,0,0]]))[0]:.4f} (expect 0.5000)")

In [ ]:
Tn = np.concatenate([load_gas_tensor(f"{DATA_DIR}/{_s}/Mtensor_Gas.npy") for _s in SNAPSHOTS], axis=0)
assert len(Tn) == len(m200), f"tensor has {len(Tn)} rows, catalogue has {len(m200)}"

q3, s3 = axis_ratios_3d(Tn)
T["ellip_3d"]        = 1.0 - s3[sel]                       # DIAGNOSTIC ONLY
T["triaxiality"]     = triaxiality(Tn)[sel]                # DIAGNOSTIC ONLY
T["ellip_projected"] = ellipticity_projected(Tn, seed=0)[sel]        # OBSERVABLE
T["ellip_proj_mean"] = ellipticity_projected_mean(Tn, 24, 0)[sel]    # diagnostic

e1 = ellipticity_projected(Tn, seed=0)
e2 = ellipticity_projected(Tn, seed=1)
print(f"3D minor/major    median {np.median(s3):.3f}  "
      f"16-84 [{np.percentile(s3,16):.3f}, {np.percentile(s3,84):.3f}]")
print(f"projected e       median {np.median(e1):.3f}  "
      f"16-84 [{np.percentile(e1,16):.3f}, {np.percentile(e1,84):.3f}]")
print(f"\ncorr(e_proj, 1-s3)      = {np.corrcoef(e1, 1-s3)[0,1]:+.3f}  "
      f"projection keeps most of the 3D signal")
print(f"corr(e_LOS0, e_LOS1)    = {np.corrcoef(e1, e2)[0,1]:+.3f}  "
      f"the rest is orientation scatter")

---
## 8. Pressure Profile and Feature Graphs

In [ ]:
P_th_prof = P_e_all * PTH_OVER_PE # TOTAL thermal pressure profile
min_r = 1


### Pressure profiles

In [ ]:

# ── Figure 1 (waterfall / offset version) ──
n_profiles = min(5, len(sel))
halo_subset = np.random.choice(sel, size=n_profiles, replace=False)
# halo_subset = inds_z0_big_halos[::max(1, len(inds_z0_big_halos) // n_profiles)][:n_profiles]

ids = np.where(qual)[0][:5]
fig, ax = plt.subplots(figsize=(7, 6))

# Offset each curve by a factor of 10^j on the log axis
offset_per_curve = 2.0   # each curve shifted by 10^offset_per_curve

for j, hid in enumerate(ids):
    try:
        i = int(T["halo_id"][hid])
        P = P_e_all[i] * PTH_OVER_PE
        f = fit_gnfw(R_EFF, P_e_all[i])
        off = 10.0 ** (j * offset_per_curve)

        lbl_true = 'true' if j == 0 else None
        lbl_fit  = 'fit'  if j == 0 else None
        
        P_norm = P / interp_profile(P, 1.0)
        ax.loglog(R_EFF, P_norm * off, color='C3',  alpha=0.7, lw=1.2, label=lbl_true)
        fit_curve = np.exp(gnfw_lnP(np.log(R_EFF), f["ln_P0"], f["ln_xc"], f["beta"]))
        fit_curve /= np.exp(gnfw_lnP(0.0, f["ln_P0"], f["ln_xc"], f["beta"]))
        ax.loglog(R_EFF, fit_curve * off, 'b--',  alpha=0.7, lw=1.2, label=lbl_fit)
        
        # Annotate the halo mass next to each curve
        ax.text(R_EFF[0], (P_norm[0]* off * 0.4),
            rf"$\log M={np.log10(T['M200c'][hid]):.2f}$, "
            rf"$s_{{200}}={T['slope_R200'][hid]:+.2f}$, "
            rf"rms$={T['gnfw_rms_resid'][hid]:.3f}$", 
            fontsize=10,
            horizontalalignment='left',
            verticalalignment='top',
            rotation=-8)
        ax.axvspan(*FIT_RANGE, color="0.88", zorder=0)
        ax.axvline(1.0, color="0.4", ls=":")    
        # ax.text(r[-1] * 1.05, (P_true * offset)[-1],
        #         rf'$\log_{{10}} M_{{200c}} = {log_mass:.2f}$',
        #         fontsize=10, verticalalignment='center')
    except RuntimeError:
        pass

ax.set_xlabel(r'$r\,/\,R_{200c}$', fontsize=16)
ax.set_ylabel(r'$P\,/\,P(R_{200c})$  (offset for clarity)', fontsize=16)
ax.legend(loc='upper right', fontsize=12)
ax.set_title(rf'{len(halo_subset)} massive halos at $z=0$', fontsize=15)
plt.tight_layout()
plt.savefig(PLOTDIR+'waterfall_halo_profiles.png', bbox_inches='tight')
plt.show()


### Simple relationship graphs

In [ ]:
plt.scatter(np.log10(T['M200c']), np.log10(T['M_hse']))
plt.xlabel(r'$\log_{10} \mathrm{M_{200c}}$')
plt.ylabel(r'$\log_{10} \mathrm{M_{HSE}}$')

In [ ]:
plt.scatter(np.log10(T['M200c']), 1 - T['M_hse']/T['M200c'], s=0.5);
plt.xlabel(r'$\log_{10}( M_\mathrm{200c})$')
plt.ylabel(r'$b = 1 - \mathrm{M_{HSE}}/\mathrm{M_{200c}}$')
plt.savefig(PLOTDIR+'mtruevshse_z0.png', bbox_inches='tight',dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Figure 16: Gas concentration vs. Hydrostatic Mass Bias
# ══════════════════════════════════════════════════════════════════

# --- Option A: Using the big-halos subset (M > MASS_CUT, z=0 only) ---
# hse_masses was computed for inds_z0_big_halos
# GasConc is defined on the full (concatenated) catalog

# Gas concentration for this subset
cgas_big = T['c_gas_0p50']

# HSE bias: (M_HSE - M_true) / M_true
hse_bias_big = 1 - T['M_hse'] / T['M200c']

plt.figure(figsize=(7, 5))
plt.scatter(cgas_big, hse_bias_big, s=2, alpha=0.8, color='C0')
plt.xlabel(r'Gas concentration $c_\mathrm{gas}$', fontsize=16)
plt.ylabel(r'$b = 1 - \frac{M_\mathrm{HSE}}{M_\mathrm{true}}$',
           fontsize=16)
plt.tight_layout()
#plt.savefig(PLOTDIR+'gascvshse_z0.png', bbox_inches='tight',dpi=150)
plt.show()

### Gas Fraction and Stellar Fraction

In [ ]:
plt.scatter(np.log10(TNG_TO_MSUN * m200[sel]), (mGas / m200[sel]), s=0.5, label='Gas')
plt.scatter(np.log10(TNG_TO_MSUN * m200[sel]), (mStar / m200[sel]), s=0.5, label='Stellar')

plt.ylabel(r'Mass fraction')
plt.xlabel(r'$\mathrm{M_{200c}}$')
plt.legend()

### Bias Overview

In [ ]:
m, b = T["M200c"][qual], T["bias"][qual]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(m, b, s=4, alpha=0.55, color='C1')
edges = np.logspace(np.log10(m.min()), np.log10(m.max()), 8)
cen   = np.sqrt(edges[:-1] * edges[1:])
med   = [np.median(b[(m >= lo) & (m < hi)]) if ((m >= lo) & (m < hi)).sum() > 5 else np.nan
         for lo, hi in zip(edges[:-1], edges[1:])]
ax[0].plot(cen, med, "k--", lw=1.7, label="median")
ax[0].axhline(0, color="0.3", lw=1)
ax[0].set_xscale("log")
ax[0].set_xlabel(r"$M_{200c}\ [M_\odot]$")
ax[0].set_ylabel(r"$b = 1 - M_{\rm HSE}/M_{200c}$")
ax[0].legend()

ax[1].hist(b, bins=45, alpha=0.85, color='C1')
ax[1].axvline(np.median(b), color="k", ls="--", label=f"median {np.median(b):+.3f}")
ax[1].axvline(0, color="0.4", lw=1)
ax[1].set_xlabel(r"$b$"); ax[1].set_ylabel("halos"); ax[1].legend()

ax[2].scatter(T["M_hse"][qual], T["M200c"][qual], s=4, alpha=0.35, color='C1')
lim = [T["M200c"][qual].min(), T["M200c"][qual].max()]
ax[2].plot(lim, lim, "k--", lw=1.5, label="1:1")
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_xlabel(r"$M_{\rm HSE}\ [M_\odot]$")
ax[2].set_ylabel(r"$M_{200c}\ [M_\odot]$"); ax[2].legend()
plt.tight_layout(); 
plt.savefig(PLOTDIR + "bias_overview.png", dpi=150,
                                bbox_inches="tight")
plt.show()

---
## 9. Decomposing $b$ definition

At `r = R200c`, hydrostatic equilibrium gives *exactly*

$$1-b=\frac{M_{\rm HSE}}{M_{200c}}
=\underbrace{\frac{kT_{200}}{\mu m_p}\frac{R_{200c}}{GM_{200c}}}_{T_{200}/T_{\rm vir}}
\times\underbrace{\left(-\frac{d\ln P}{d\ln r}\bigg|_{R_{200c}}\right)}_{-s_{200}}$$

So the bias is *how hot the gas is at R200c relative to virial* times *how
steeply the pressure falls there*. Nothing else. **The modelling problem is to
predict those two factors from observables**.

Panel 1 must be 1:1 with essentially zero scatter. 

In [ ]:
kT_erg = T["kT_R200"][qual] * KEV_ERG
R_cm   = T["R200c"][qual] * T["a_scale"][qual] * TNG_TO_CM   # physical radius = comoving * a=1/(1+z)
M_g    = T["M200c"][qual] * MSUN_G
T_over_Tvir = kT_erg * R_cm / (MU * PROTON_TO_G * G_CGS * M_g)
predicted   = T_over_Tvir * (-T["slope_R200"][qual])

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(predicted, 1 - T["bias"][qual], s=4, alpha=0.35, color='C1')
lim = [predicted.min(), predicted.max()]
ax[0].plot(lim, lim, "k--", lw=1.5)
ax[0].set_xlabel(r"$(T_{200}/T_{\rm vir})\times(-s_{200})$")
ax[0].set_ylabel(r"$1-b$"); ax[0].set_title("exact identity")

ax[1].scatter(T["slope_R200"][qual], T["bias"][qual], s=4, alpha=0.35, color='C1')
ax[1].set_xlabel(r"$d\ln P/d\ln r|_{R_{200c}}$"); ax[1].set_ylabel(r"$b$")
ax[1].set_title(f"r = {np.corrcoef(T['slope_R200'][qual], T['bias'][qual])[0,1]:+.3f}")

ax[2].scatter(T_over_Tvir, T["bias"][qual], s=4, alpha=0.35, color='C1')
ax[2].set_xlabel(r"$T_{200}/T_{\rm vir}$"); ax[2].set_ylabel(r"$b$")
ax[2].set_title(f"r = {np.corrcoef(T_over_Tvir, T['bias'][qual])[0,1]:+.3f}")
plt.tight_layout()
plt.savefig(PLOTDIR + "b_identity.png", dpi=150, bbox_inches="tight")
plt.show()

resid = (1 - T["bias"][qual]) - predicted
print(f"identity residual: mean {np.mean(resid):+.2e}, max |resid| "
      f"{np.max(np.abs(resid)):.2e}   (should be ~machine precision)")
T["T_over_Tvir"] = np.nan * np.ones(len(sel)); T["T_over_Tvir"][qual] = T_over_Tvir

---
## 10. Chosen and Freeze gNFW fit range

`FIT_RANGE` is a choice, and it moves `b`. Pick it on physical grounds (exclude
the AGN-shaped core, keep enough lever arm at R200c), check it here, then
**freeze it**. Choosing it later, after seeing which value helps a candidate
equation, is a garden of forking paths.

Previously used 0.03 to 2.50 (actually slightly off due to resolved bin mismatching)

Can use this diagnostic: the range that minimises disagreement between the gNFW
slope and the model-free local slope, since that is the range over which the
three-parameter model is actually describing the data.

In [ ]:
sub = np.where(qual)[0][:400]
hid_s = T["halo_id"][sub].astype(int)

print(f"{'fit range':<16} {'median b':>10} {'scatter':>9} "
      f"{'median rms':>11} {'|slope_fit - slope_loc|':>24}")
print("-" * 76)
for lo, hi in [(0.03, 2.50), (0.05, 2.20), (0.10, 2.00),
               (0.15, 1.80), (0.20, 2.00), (0.30, 2.00)]:
    bs, rs, ds = [], [], []
    for i in hid_s:
        f = fit_gnfw(R_EFF, P_e_all[i], fit_range=(lo, hi))
        if not f["converged"]:
            continue
        s_fit = float(gnfw_logslope(0.0, f["ln_xc"], f["beta"]))
        s_loc = local_log_slope(R_EFF, P_e_all[i])
        P_e_r, rho_r = interp_profile(P_e_all[i], 1.0), interp_profile(rho_all[i], 1.0)
        m = -(r200[i]*a_scale_all[i]*TNG_TO_CM*P_e_r*PTH_OVER_PE/(G_CGS*rho_r))*s_fit/MSUN_G
        bs.append(1 - m/(m200[i]*TNG_TO_MSUN)); rs.append(f["rms"])
        ds.append(abs(s_fit - s_loc))
    print(f"[{lo:.2f}, {hi:.2f}]{'':<5} {np.median(bs):>+10.4f} {np.std(bs):>9.4f} "
          f"{np.median(rs):>11.4f} {np.median(ds):>24.4f}")

---
## 11. Feasability and Usability of Observables

The deliverable is a formula someone can *apply*. An observer measures
`M_HSE` and wants `M_true`; **they do not know `M_200c`.** So the usable form is

$$M_{\rm true}=\frac{M_{\rm HSE}}{1-b(\text{observables})}$$

Observables can be sepparated into four tiers:

| tier | meaning |
|---|---|
| `DIRECT` | routinely measured today (SZ, X-ray, optical) |
| `FEASIBLE` | possible with current instruments, but hard — usually needs resolved profiles to R200c |
| `PROJECTED` | simulation gives 3-D; project before using to mimic observation|
| `SIM_ONLY` | not observable — diagnostic only, never in an equation |

In [ ]:
OBSERVABILITY_HSE = {
    "M_hse":            dict(tier="ROUTINE-DERIVED",   premise="STRUCTURAL"),
    "slope_R200":       dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "slope_R200_0p65":  dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "slope_local_R200": dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "kT_R200":          dict(tier="DEMONSTRATED-HARD", premise="IDEALIZED-STAND-IN"),
    "gnfw_rms_resid":   dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "beta":             dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "xc":               dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "c_Y_0p50":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "c_Y_0p15":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "logit_c_Y_0p50":   dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "c_gas_0p50":       dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "c_gas_0p15":       dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "Y_ratio_0p65":     dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "Y_X_500":          dict(tier="ROUTINE-DERIVED",   premise="NON-PHYSICAL-IDEALIZED-STAND-IN"), # No core excised!! Also uses fake R500c, so not included.
    "Y_X_200":          dict(tier="DEMONSTRATED-HARD", premise="IDEALIZED-STAND-IN"), # Non core excised!!
    "Y200_HSE":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "Y500_HSE":         dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "M_gas":            dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "M_star":           dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "Mstar_over_Mgas":  dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "ellip_projected":  dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "f_gas_hse":        dict(tier="ROUTINE-DERIVED",   premise="STRUCTURAL"),
    "M200c":            dict(tier="SIMULATION",       premise=None),
    "R200c":            dict(tier="SIMULATION",       premise=None),
    "f_gas_true":       dict(tier="SIMULATION",       premise=None),
    "T_over_Tvir":      dict(tier="SIMULATION",       premise=None),
    "ellip_3d":         dict(tier="SIMULATION",       premise=None),
    "triaxiality":      dict(tier="SIMULATION",       premise=None),
    "ellip_proj_mean":  dict(tier="SIMULATION",       premise=None),
}
REDUNDANT_HSE = {"beta": "slope_R200", "xc": "slope_R200", "logit_c_Y_0p50": "c_Y_0p50"}   # exact-duplicate shape params
HSE_BASE_VARS = {"slope_R200", "slope_R200_0p65", "slope_local_R200", "f_gas_hse", "kT_R200", "M_hse"}                   # amplitude of the relation, not predictors

CANDIDATE_FEATURES_HSE_OBS = [
    k for k, v in OBSERVABILITY_HSE.items()
    if v["premise"] == "PHYSICAL"
    and v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_HSE
    and k not in HSE_BASE_VARS
    and k in T
]

CANDIDATE_FEATURES_HSE_OBS_THEORY = [
    k for k, v in OBSERVABILITY_HSE.items()
    if v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_HSE
    and k not in HSE_BASE_VARS
    and k in T
]

CANDIDATE_FEATURES_HSE_THEORY = [
    k for k, v in OBSERVABILITY_HSE.items()
    if v["premise"] != "PHYSICAL"
    and v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_HSE
    and k not in HSE_BASE_VARS
    and k in T
]

CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL = [
    k for k, v in OBSERVABILITY_HSE.items()
    if v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_HSE
    and k in T
]

print(f"HSE candidate pool OBS: {len(CANDIDATE_FEATURES_HSE_OBS)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_HSE_OBS))

print(f"HSE candidate pool OBS + THEORY: {len(CANDIDATE_FEATURES_HSE_OBS_THEORY)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_HSE_OBS_THEORY))

print(f"HSE candidate pool THEORY: {len(CANDIDATE_FEATURES_HSE_THEORY)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_HSE_THEORY))


print(f"HSE candidate pool THEORY + TRIVIAL: {len(CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL))

print(f"excluded as base variables of the relation: {sorted(HSE_BASE_VARS & set(OBSERVABILITY_HSE))}")
print(f"excluded as exact-duplicate shape params:   {sorted(REDUNDANT_HSE)}")

### Finite + Quality Mask: USE

In [ ]:
ALL_FEATURES_EVER = sorted(set(
    [c for c in CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL if c in T] + ["M200c", "bias"]
))

_finite = np.ones(qual.sum(), dtype=bool)
for c in ALL_FEATURES_EVER:
    v = np.asarray(T[c][qual], dtype=float)
    _finite &= np.isfinite(v)

USE = np.where(qual)[0][_finite]          # absolute indices into T
N_USE = len(USE)
print(f"{qual.sum()} pass quality, {N_USE} finite in every feature "
      f"({qual.sum() - N_USE} dropped)")

### Corner Plot

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Corner plot: pairwise feature distributions
# Diagonal  = 1D marginal histogram of each feature
# Lower tri = 2D density (hist2d) with contours, + scattered tail points
# ══════════════════════════════════════════════════════════════════

# 1. Map the dictionary keys in 'T' to their display labels.
# (Adjust this list to include/exclude whatever features you want)
# features_to_plot = [
#     ("M200c",           r'$M_{200c}$'),
#     ("c_Y_0p50",        r'$c_Y^{(0.5)}$'),
#     ("Mstar_over_Mgas", r'$M_*/M_\mathrm{gas}$'),
#     ("c_gas_0p50",      r'$c_\mathrm{gas}^{(0.5)}$'),
#     ("c_Y_0p25",        r'$c_Y^{(0.25)}$'),
#     ("Y_ratio_0p65",    r'$c_Y^{(0.15/0.65)}$'), # adjust key if needed
#     ("c_Y_0p15",        r'$c_Y^{(0.15)}$'),
#     ("beta",            r'$\beta$'),
#     ("xc",              r'$x_c$'),
#     ("c_gas_0p15",      r'$c_\mathrm{gas}^{(0.15)}$'),
#     ("slope_R200",      r'slope$@R_{200c}$'),
#     ("slope_local_R200",r'slope$@R_{500c}$'),
#     ("bias",            r'HSE bias')
# ]
keys = [c for c in CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL if c in T]
n_feat = len(keys)

# 2. Filter for good data
# Start with the 'qual' mask you defined in the previous cell
valid_rows = qual.copy()

# Ensure all selected features contain finite numbers (no NaNs or infs)
for k in keys:
    valid_rows &= np.isfinite(T[k])

# 3. Stack the 1D arrays from T back into a 2D matrix X
# This mimics the old 'X' structure so the plotting loop doesn't have to change
X = np.column_stack([T[k][valid_rows] for k in keys])

print(f'Corner plot on {X.shape[0]} halos, {n_feat} features')

fig, axes = plt.subplots(n_feat, n_feat, figsize=(2.0 * n_feat, 2.0 * n_feat))
fig.subplots_adjust(hspace=0.05, wspace=0.05)

for i in range(n_feat):
    for j in range(n_feat):
        ax = axes[i, j]

        # Upper triangle: hide
        if j > i:
            ax.axis('off')
            continue

        xi = X[:, j]
        yi = X[:, i]

        # Robust axis ranges (trim extreme outliers so bulk is visible)
        xlo, xhi = np.percentile(xi, [1, 99])
        ylo, yhi = np.percentile(yi, [1, 99])

        if i == j:
            # Diagonal: 1D marginal histogram
            ax.hist(xi, bins=30, range=(xlo, xhi), color='C0', alpha=0.7)
            ax.set_yticks([])
        else:
            # Lower triangle: 2D density + contours + tail scatter
            ax.hist2d(xi, yi, bins=40, range=[[xlo, xhi], [ylo, yhi]],
                      cmap='Blues', cmin=1)
            # Light scatter for the sparse tail points (optional visual aid)
            ax.scatter(xi, yi, s=1, color='k', alpha=0.05, rasterized=True)
            ax.set_xlim(xlo, xhi)
            ax.set_ylim(ylo, yhi)

        # Tick label management (only edges)
        if i < n_feat - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel(keys[j], fontsize=12)
            ax.tick_params(axis='x', labelrotation=45, labelsize=7)
        if j > 0 or i == 0:
            ax.set_yticklabels([])
        else:
            ax.set_ylabel(keys[i], fontsize=12)
            ax.tick_params(axis='y', labelsize=7)
        # Leftmost column y-label even on diagonal-free rows
        if j == 0 and i != 0:
            ax.set_ylabel(keys[i], fontsize=12)

plt.suptitle('Pairwise feature distributions (HSE bias features)',
             fontsize=15, y=0.92)
# plt.savefig(plotdir+'hse_feature_corner.png', bbox_inches='tight', dpi=150)
plt.show()

### Gas Fraction Baseline and Justification for Exclusion

Your MTNG measurement is `σ(ln f_gas_true) = 0.051`, while `σ(ln(1−b)) = 0.160`
— 3.1× larger. The signal-to-noise of the inversion is just that ratio. I swept it:
 
| σ(ln f_gas) | regime | variance explained | RMS in b |
|---|---|---|---|
| **0.051** | **MTNG z=0 (yours)** | **89.1%** | **0.042** |
| 0.10 | cross-simulation spread (TNG/EAGLE) | 56.8% | 0.083 |
| 0.15 | X-ray obs., massive relaxed | **−2.3%** | 0.128 |
| 0.25 | X-ray obs., full range | −192% | 0.216 |
 
**At σ = 0.15 the method is already worse than predicting the median.** Your 89%
is a measurement of MTNG's feedback model being unusually uniform, not a
statement about clusters. One simulation, one feedback prescription, one
redshift, and a mass range where the baryon fraction has nearly saturated.

In [ ]:
f_true = T["f_gas_true"][USE]
f_hse  = T["f_gas_hse"][USE]
bb     = T["bias"][USE]

print("f_gas_hse = f_gas_true / (1 - b)\n")
print(f"  scatter in ln f_gas_true : {np.std(np.log(f_true)):.4f}")
print(f"  scatter in ln (1 - b)    : {np.std(np.log(1-bb)):.4f}   "
      f"<-- {np.std(np.log(1-bb))/np.std(np.log(f_true)):.1f}x larger")
print(f"  corr(f_gas_hse, b)       = {np.corrcoef(f_hse, bb)[0,1]:+.3f}")

c0 = np.median(f_true)
b_fgas = 1 - c0 / f_hse
print(f"\n  ONE-PARAMETER model  b = 1 - {c0:.4f}/f_gas_hse")
print(f"    MSE {np.mean((b_fgas-bb)**2):.5f}   var(b) {np.var(bb):.5f}   "
      f"variance explained {1-np.mean((b_fgas-bb)**2)/np.var(bb):.1%}")
print("\n  => This is the classical f_gas mass-calibration method, not a new")
print("     result. Treat it as a BASELINE to compare against, not a feature.")
print("     Caveat: real clusters scatter in f_gas more than the simulation,")
print("     so this baseline is optimistic.")


--- 
## 12. Feature Selection

In [ ]:
from itertools import combinations
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold


def _est(seed=0):
    return GradientBoostingRegressor(n_estimators=150, max_depth=3,
                                     learning_rate=0.05, subsample=0.8,
                                     random_state=seed)


def _wmean(x, w):
    return float(np.sum(w * x) / np.sum(w))


def wvar(y, w=None):
    """Weighted variance -- the baseline that matches cv_score's weighted MSE."""
    y = np.asarray(y, float)
    if w is None:
        return float(np.var(y))
    w = np.asarray(w, float)
    mu = np.sum(w * y) / np.sum(w)
    return float(np.sum(w * (y - mu) ** 2) / np.sum(w))


def cv_score(X, y, w=None, n_splits=5, seed=0):
    """K-fold CV MSE, weighted CONSISTENTLY with the fit. Deliberately a weak
    learner: we measure how much signal a feature SET carries, not the best
    model. When sample weights are given BOTH the fit and the scored MSE are
    weighted -- scoring an unweighted MSE against a weighted fit (the old bug)
    silently rewards features that track the low-weight bulk instead of the
    high-mass end the weighting was meant to emphasize."""
    X = np.atleast_2d(X)
    if X.shape[0] != len(y):
        X = X.T
    errs = []
    for tr, te in KFold(n_splits, shuffle=True, random_state=seed).split(X):
        m = _est(seed)
        m.fit(X[tr], y[tr], sample_weight=None if w is None else w[tr])
        se = (m.predict(X[te]) - y[te]) ** 2
        errs.append(np.mean(se) if w is None else _wmean(se, w[te]))
    return float(np.mean(errs))


CANDIDATE_FEATURES_HSE = [c for c in CANDIDATE_FEATURES_HSE_OBS_THEORY if c in T]

A = np.column_stack([T[c][USE] for c in CANDIDATE_FEATURES_HSE])
rho = np.atleast_2d(spearmanr(A).statistic)

THRESH = 0.90
unassigned, groups = set(range(len(CANDIDATE_FEATURES_HSE))), []
while unassigned:
    i = min(unassigned)
    g = {i} | {j for j in unassigned if j != i and abs(rho[i, j]) >= THRESH}
    groups.append(sorted(g)); unassigned -= g

print(f"redundancy groups at |spearman| >= {THRESH} (HSE pool):")
for g in groups:
    nm = [CANDIDATE_FEATURES_HSE[i] for i in g]
    print("   " + (" ~ ".join(nm) if len(nm) > 1 else nm[0]))
print("\nSR will pick one member of each group essentially at random, and a")
print("rerun with a different seed picks another. Keep ONE per group.")


In [ ]:
y_all = np.asarray(T["bias"][USE], float)
w_all = np.asarray(T["M200c"][USE] * T["E_z"][USE]**(-0.4), float)   # denominator-rule weight: evolution-scaled true mass
base_var = wvar(y_all, w_all)

uni = []
for c in CANDIDATE_FEATURES_HSE:
    v = np.asarray(T[c][USE], float)
    m = cv_score(v[:, None], y_all, w_all)
    uni.append((c, m, 1 - m/base_var, float(spearmanr(v, y_all).statistic)))
uni.sort(key=lambda r: r[1])

print(f"var(b) = {base_var:.5f}   (the number to beat)\n")
print(f"{'feature':<22} {'CV MSE alone':>13} {'frac var':>10} {'spearman':>10}")
print("-"*58)
for c, m, fv, sp in uni:
    print(f"{c:<22} {m:>13.5f} {fv:>10.3f} {sp:>+10.3f}")
print("\nA feature with frac_var <= 0 carries no signal ON ITS OWN. It may still")
print("be a useful correction inside a larger model -- but not inside a 10-token")
print("equation, which is what we are building.")

In [ ]:


# PRUNED = [
#     "M_hse", "slope_R200", "kT_R200", "c_Y_0p50", "c_gas_0p50",
#     "Mstar_over_Mgas", "gnfw_rms_resid", "ellip_projected", "c_Y_0p15",
# ]
# PRUNED = [c for c in PRUNED if c in T]

subset_results = {}
for size in (1, 2, 3, 4, 5):
    rows = []
    for combo in combinations(CANDIDATE_FEATURES_HSE, size):
        M = np.column_stack([T[c][USE] for c in combo])
        rows.append((combo, cv_score(M, y_all, w_all)))
    rows.sort(key=lambda r: r[1])
    subset_results[size] = rows
    print(f"\n--- best subsets of size {size} ---")
    for combo, m in rows[:5]:
        print(f"   {1-m/base_var:>6.3f} frac var   MSE {m:.5f}   {' + '.join(combo)}")

best_by_size = {s: r[0] for s, r in subset_results.items()}
print("\nPLATEAU DIAGNOSTIC — where does adding a feature stop paying?")
prev = base_var
for s in (1, 2, 3, 4, 5):
    m = best_by_size[s][1]
    print(f"   size {s}: frac var {1-m/base_var:.3f}   gain over size {s-1}: "
          f"{100*(prev-m)/base_var:+.1f}% of var(b)")
    prev = m
print("\nThe size at which the gain drops below ~2% of var(b) is the number of")
print("features your analytic equation should contain. More than that and SR")
print("is fitting noise with extra tokens.")

In [ ]:
SIZE, N_BOOT = 3, 30
_rng_boot = np.random.default_rng(0)
counts = {c: 0 for c in CANDIDATE_FEATURES_HSE}
combo_counts = {}

for b_i in range(N_BOOT):
    sub = _rng_boot.choice(N_USE, size=int(0.7*N_USE), replace=False)
    best, best_m = None, np.inf
    for combo in combinations(CANDIDATE_FEATURES_HSE, SIZE):
        M = np.column_stack([T[c][USE][sub] for c in combo])
        m = cv_score(M, y_all[sub], w_all[sub], n_splits=3, seed=b_i)
        if m < best_m:
            best, best_m = combo, m
    for c in best:
        counts[c] += 1
    combo_counts[best] = combo_counts.get(best, 0) + 1

print(f"selection frequency over {N_BOOT} resamples (subsets of size {SIZE}):")
for c, k in sorted(counts.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(30*k/N_BOOT)
    print(f"   {c:<22} {k/N_BOOT:>5.0%}  {bar}")
print("\nwinning combinations:")
for combo, k in sorted(combo_counts.items(), key=lambda kv: -kv[1])[:5]:
    print(f"   {k/N_BOOT:>5.0%}  {' + '.join(combo)}")

---
## 13. RF

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

#rng = np.random.default_rng(42)

### Define Feature Sets

In [ ]:
feat_conc = [c for c in ["c_Y_0p50", "c_Y_0p15",
                        "c_gas_0p50", "c_gas_0p15", "Y_ratio_0p65"] if c in T]

# feat_obs_no_slope = [c for c in ["M_hse", "kT_R200", "c_Y_0p50", "c_Y_0p15",
#                         "c_gas_0p50", "c_gas_0p15", "Mstar_over_Mgas",
#                         "gnfw_rms_resid", "ellip_projected"] if c in T]

feat_obs = CANDIDATE_FEATURES_HSE_OBS

feat_obs_theory = CANDIDATE_FEATURES_HSE_OBS_THEORY

feat_trivial = CANDIDATE_FEATURES_HSE_OBS_THEORY_TRIVIAL




In [ ]:
# One split, one seed, its own generator — NOT the `rng` consumed in the unit
# check cell, whose state depends on how many times you ran that cell.
SPLIT_SEED = 20260101
_split_rng = np.random.default_rng(SPLIT_SEED)
# Stratified 50/50 split on the evolution-scaled mass S = M200c * E(z)^(-2/5),
# so both snapshots and the whole mass range are balanced across train/test.
_S = T["M200c"][USE] * T["E_z"][USE]**(-0.4)
_NBIN_STRAT = 10
_qedges = np.quantile(_S, np.linspace(0, 1, _NBIN_STRAT + 1))
_qedges[0] *= (1 - 1e-9); _qedges[-1] *= (1 + 1e-9)
maskTest = np.zeros(N_USE, dtype=bool)
for _bi in range(_NBIN_STRAT):
    _inb = np.where((_S >= _qedges[_bi]) & (_S < _qedges[_bi + 1]))[0]
    _perm = _split_rng.permutation(_inb)
    maskTest[_perm[len(_perm) // 2:]] = True
print(f"train {(~maskTest).sum()}  test {maskTest.sum()}  seed {SPLIT_SEED}  "
      f"(stratified on M200c*E(z)^-2/5, {_NBIN_STRAT} bins)")
for _s in SNAPSHOTS:
    _mm = (T["z"][USE] == float(_s.split("=")[1]))
    print(f"   {_s}: train {(_mm & ~maskTest).sum()}  test {(_mm & maskTest).sum()}")

MASS_TEST = T["M200c"][USE][maskTest]     # always aligned by construction

def make_xy(feats):
    """Aligned to USE for every feature set. Never returns a different length."""
    X = np.column_stack([np.asarray(T[c][USE], float) for c in feats])
    y = np.asarray(T["bias"][USE], float)
    w = np.asarray(T["M200c"][USE] * T["E_z"][USE]**(-0.4), float)   # evolution-scaled true mass
    return X, y, w

# MASS_EDGES=np.logspace(13.825,14.80,num=6) #(3.6,4.7,num=7);
MASS_EDGES = np.logspace(np.log10(MASS_TEST.min()*1.001),
                         np.log10(np.percentile(MASS_TEST, 99)), 6)

xbins=np.sqrt(MASS_EDGES[:-1]*MASS_EDGES[1:]);


def rf_resid(feats, label, color):
    X, y, w = make_xy(feats)
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                               max_depth=None, random_state=0, n_jobs=-1)
    # rf = RandomForestRegressor(n_estimators=50, min_samples_leaf=5,
    #                            max_depth=50, random_state=0, n_jobs=-1)
    rf.fit(X[~maskTest], y[~maskTest], sample_weight=w[~maskTest])
    resid_rf = rf.predict(X[maskTest]) - y[maskTest]
    b_pred = rf.predict(X[maskTest])
    resid_rf_m = (b_pred - y[maskTest]) / (1 - b_pred)
    return dict(resid=resid_rf, resid_m=resid_rf_m,
                label=label, color=color, rf=rf, X=X, y=y)


y_tr = np.asarray(T["bias"][USE], float)[~maskTest]
y_te = np.asarray(T["bias"][USE], float)[maskTest]

models = {}
# BASELINE: no model at all. Its residual scatter IS the bias scatter, so every
# ratio below reads as "fraction of the bias scatter that survives".
models["none"] = dict(resid=np.median(y_tr) - y_te, 
                    resid_m=(np.median(y_tr) - y_te) / (1 - np.median(y_tr)),
                    label=r"Median $b$", color="C0")
# The classical f_gas method, as a named competitor rather than a feature.
_c0 = np.median(T["f_gas_true"][USE][~maskTest])


models["fgas"] = dict(resid=(1 - _c0/T["f_gas_hse"][USE][maskTest]) - y_te,
                    resid_m=((1 - _c0/T["f_gas_hse"][USE][maskTest]) - y_te) / (1 - (1 - _c0/T["f_gas_hse"][USE][maskTest])),
                      label=r"$f_\mathrm{gas}$ method", color="C1")
models["conc"] = rf_resid(feat_conc,
                           r"RF[$M_\mathrm{HSE}, c_Y, c_\mathrm{gas}$]", "C2")
models["obs"]  = rf_resid(feat_obs, "RF[all observables]", "C3")
models["obs_thry"]  = rf_resid(feat_obs_theory, "RF[all observables + theory]", "C4")
models["trivial"]  = rf_resid(feat_trivial, "RF[trivial]", "C5")



In [ ]:

def binned_stats(mass, resid, edges, n_boot=500, seed=0):
    rng_b = np.random.default_rng(seed)
    nb = len(edges) - 1
    out = {k: np.full(nb, np.nan) for k in
           ("std", "std_err", "mean", "mean_err")}
    out["count"] = np.zeros(nb, dtype=int)
    
    for i in range(nb): # Compute std for each run in mass bins defined by 'temp = np.logspace..'
        mask = (mass >= edges[i]) & (mass < edges[i+1]) & np.isfinite(resid)
        out["count"][i] = len(resid[mask])
        if len(resid[mask]) < 10:
            continue
        out['std'][i]=np.std(resid[mask])
        out['mean'][i] = np.mean(resid[mask])
        bs = resid[mask][rng_b.integers(0, len(resid[mask]), size=(n_boot, len(resid[mask])))]
        out["std_err"][i] = np.std(np.std(bs, axis=1))
        out["mean_err"][i] = np.std(np.mean(bs, axis=1))
    
    out["centers"] = np.sqrt(edges[:-1]*edges[1:])
    return out


stats = {k: binned_stats(MASS_TEST, v["resid"], MASS_EDGES)
         for k, v in models.items()}
stats_m = {k: binned_stats(MASS_TEST, v["resid_m"], MASS_EDGES)
         for k, v in models.items()}
base = stats["none"]
NOISE_FLOOR = 0.04          # from the synthetic fit-induced-bias test


### Check RF Hyperparams

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL A: Weighting functions (add near top of notebook)
# ═══════════════════════════════════════════════════════════════

def weighted_mse(y_true, y_pred, weights):
    """Weighted mean squared error."""
    return np.average((y_true - y_pred)**2, weights=weights)

def kde_inverse_density_weights(masses, bandwidth=0.15):
    """
    KDE-based inverse-density weighting in log-mass space.
    Each mass range contributes equally to the metric.
    """
    log_m = np.log10(masses)
    kde = gaussian_kde(log_m, bw_method=bandwidth)
    density = kde(log_m)
    w = 1.0 / density
    return w / w.sum() * len(w)  # normalize so mean weight = 1


# ═══════════════════════════════════════════════════════════════
# CELL B: Learning curves with top-heavy evaluation
# ═══════════════════════════════════════════════════════════════

# --- Evaluation weights for test/train error ---
# These weight the METRIC, not the training. Training still uses sample_weight=pow(m200H,1).
test_masses = T['M200c'][USE][maskTest]
train_masses = T['M200c'][USE][~maskTest]

Xo, yo, wo = make_xy(feat_obs)

# Three evaluation schemes:
#   1. Uniform (original MSE)
#   2. pow(m, 3) — aggressive top-heavy
#   3. Top quartile only
mass_threshold_75 = np.percentile(T['M200c'][USE], 75)

eval_schemes = {
    'Uniform MSE': {
        'train_w': np.ones(len(train_masses)),
        'test_w':  np.ones(len(test_masses)),
    },
    r'$m^3$-weighted MSE': {
        'train_w': train_masses**3,
        'test_w':  test_masses**3,
    },
    'Top 25\% only': {
        'train_w': np.where(train_masses >= mass_threshold_75, 1.0, 0.0),
        'test_w':  np.where(test_masses >= mass_threshold_75, 1.0, 0.0),
    },
}

# ── max_depth sweep ──
depths = [3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 30, 50]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (scheme_name, weights) in zip(axes, eval_schemes.items()):
    train_err = []
    test_err = []
    for d in depths:
        rf = RandomForestRegressor(
            max_depth=d, random_state=0, min_samples_leaf=5,
            n_estimators=50, n_jobs=-1
        )
        rf.fit(Xo[~maskTest], yo[~maskTest], sample_weight=wo[~maskTest])
        train_err.append(weighted_mse(yo[~maskTest], rf.predict(Xo[~maskTest]), weights['train_w']))
        test_err.append(weighted_mse(yo[maskTest], rf.predict(Xo[maskTest]), weights['test_w']))

    ax.plot(depths, train_err, 'o-', label='Train', markersize=5)
    ax.plot(depths, test_err, 's-', label='Test', markersize=5)
    ax.set_xlabel('max\_depth')
    ax.set_ylabel('Weighted MSE')
    ax.set_title(scheme_name)
    ax.legend()

plt.tight_layout()
plt.show()

# ── min_samples_leaf sweep ──
leaf_sizes = [2, 5, 8, 10, 15, 20, 30, 40, 50, 60, 80]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (scheme_name, weights) in zip(axes, eval_schemes.items()):
    train_err = []
    test_err = []
    for ml in leaf_sizes:
        rf = RandomForestRegressor(
            max_depth=50, random_state=0, min_samples_leaf=ml,
            n_estimators=50, n_jobs=-1
        )
        rf.fit(Xo[~maskTest], yo[~maskTest], sample_weight=wo[~maskTest])
        train_err.append(weighted_mse(yo[~maskTest], rf.predict(Xo[~maskTest]), weights['train_w']))
        test_err.append(weighted_mse(yo[maskTest], rf.predict(Xo[maskTest]), weights['test_w']))

    ax.plot(leaf_sizes, train_err, 'o-', label='Train', markersize=5)
    ax.plot(leaf_sizes, test_err, 's-', label='Test', markersize=5)
    ax.set_xlabel('min\_samples\_leaf')
    ax.set_ylabel('Weighted MSE')
    ax.set_title(scheme_name)
    ax.legend()

plt.tight_layout()
plt.show()

# ── Sample weight power sweep ──
powers = [0, 0.5, 1, 1.5, 2, 2.5, 3, 4]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (scheme_name, weights) in zip(axes, eval_schemes.items()):
    train_err = []
    test_err = []
    for p in powers:
        rf = RandomForestRegressor(
            max_depth=50, random_state=0, min_samples_leaf=5,
            n_estimators=50, n_jobs=-1
        )
        sw = pow(T["M_hse"][USE], p)
        rf.fit(Xo[~maskTest], yo[~maskTest], sample_weight=sw[~maskTest])
        train_pred = rf.predict(Xo[~maskTest])
        test_pred = rf.predict(Xo[maskTest])
        
        train_err.append(np.average((yo[~maskTest] - train_pred)**2, weights=weights['train_w']))
        test_err.append(np.average((yo[maskTest] - test_pred)**2, weights=weights['test_w']))

    ax.plot(powers, train_err, 'o-', label='Train', markersize=6)
    ax.plot(powers, test_err, 's-', label='Test', markersize=6)
    ax.set_xlabel('sample\_weight power')
    ax.set_ylabel('Weighted MSE')
    ax.set_title(scheme_name)
    ax.legend()

plt.tight_layout()
plt.show()

### Plot RF Models

In [ ]:
fig = plt.figure(num=None, figsize=(5.3, 7))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

ax1 = fig.add_axes([0.15, 0.66, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax2 = fig.add_axes([0.15, 0.33, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax3 = fig.add_axes([0.15, 0.0, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(0.15,1.25))

ax1.set_yscale('linear')
ax1.set_xscale('log')
ax2.set_yscale('linear')
ax2.set_xscale('log')
ax1.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax2.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax3.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0, 1.2])

# Top panel: Y-only RF residuals as baseline
ax1.scatter(MASS_TEST, models["none"]["resid"], alpha=.8, s=6, color='C1') #ax1.scatter(m200_rf[maskTest]*1e10, bResid_M_only_rf, alpha=.8, s=6, color='C1')
ax1.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax1.errorbar(stats["none"]["centers"],stats["none"]["mean"],yerr=stats["none"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
#ax1.plot(stats["none"]["centers"],stats["none"]["mean"],'--',color='black',alpha=0.5, dashes=[5, 3])
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Middle panel: best multi-feature RF residuals
ax2.scatter(MASS_TEST, models["obs"]["resid"], alpha=.8, s=6, color='C1')
ax2.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax2.errorbar(stats["obs"]["centers"],stats["obs"]["mean"],yerr=stats["obs"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Bottom panel: relative scatter
ax3.semilogx(xbins, np.ones(len(xbins)), color='C0') #blue

for k, v in models.items():
    if k == "none":
        continue
    st = stats[k]
    ratio = st["std"]/base["std"]
    err = ratio*np.sqrt((st["std_err"]/st["std"])**2
                        + (base["std_err"]/base["std"])**2)
    ax3.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)
ax3.fill_between(base["centers"], 0, NOISE_FLOOR/base["std"], color="0.85", zorder=0)

# ax3.semilogx(xbins, rfResults["MassConc"]["std"]/rfResults["Mass"]["std"], color='C1') #orange
# ax3.semilogx(xbins, rfResults["ObsNoSlope"]["std"]/rfResults["Mass"]["std"], color='C2') #green
# ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C3') #red
#ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C4') #purple

# Axis labels — updated for bias prediction
# fig.text(0.66, 0.95, r"$b^{(1)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c}]$", rotation=0, fontsize=16)
# fig.text(0.23, 0.62, r"$b^{(2)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}]$",
#           rotation=0, fontsize=16)
fig.text(0.78, 0.95, models["none"]["label"], rotation=0, fontsize=16)
fig.text(0.62, 0.62, models["obs"]["label"], rotation=0, fontsize=16)

fig.text(-0.02, 0.75, r"$b_\mathrm{pred} - b_\mathrm{true}$", rotation=90, fontsize=18)
fig.text(-0.02, 0.4, r"$b_\mathrm{pred} - b_\mathrm{true}$", rotation=90, fontsize=18)
fig.text(0.01, 0.12, r"rel. scatter", rotation=90, fontsize=18)

# Legend annotations for scatter panel — adjust positions to match your curves
text_locations = [
    (0.64, 0.24), 
    (0.64, 0.02), 
    (0.64, 0.28), 
    (0.64, 0.20), 
    (0.64, 0.16), 
    (0.64, 0.04), 

]
for loc, v, in zip(text_locations, models.values()):
    fig.text(loc[0],loc[1], v["label"], fontsize=15, color=v["color"])

# fig.text(0.64, 0.28, r"RF[$M_\mathrm{200c}$]", fontsize=15, color='C0')
# fig.text(0.64, 0.23, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}$]", fontsize=15, color='C1')
# fig.text(0.65, 0.195, r"RF[$M_\mathrm{200c},M_*/M_\mathrm{gas}$]", fontsize=16, color='C2')
# fig.text(0.54, 0.17, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c$]", fontsize=15, color='C2')
# fig.text(0.48, 0.13, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}$]", fontsize=15, color='C3')
# fig.text(0.40, 0.02, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}$]", fontsize=15, color='C4')

plt.xlabel(r'$M_\mathrm{200c}[h^{-1} M_\odot]$', fontsize=19)
fig.text(0.5, 1, r'$z=0$', fontsize=19)

plt.savefig(PLOTDIR+'scatter_hse_rf_z0z05.png', bbox_inches='tight', dpi=150)


In [ ]:
fig = plt.figure(num=None, figsize=(5.3, 7))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

ax1 = fig.add_axes([0.15, 0.66, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax2 = fig.add_axes([0.15, 0.33, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax3 = fig.add_axes([0.15, 0.0, 0.84, 0.33], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(0.15,1.25))

ax1.set_yscale('linear')
ax1.set_xscale('log')
ax2.set_yscale('linear')
ax2.set_xscale('log')
ax1.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax2.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax3.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0, 1.2])

# Top panel: Y-only RF residuals as baseline
ax1.scatter(MASS_TEST, models["none"]["resid_m"], alpha=.8, s=6, color='C1') #ax1.scatter(m200_rf[maskTest]*1e10, bResid_M_only_rf, alpha=.8, s=6, color='C1')
ax1.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax1.errorbar(stats_m["none"]["centers"],stats_m["none"]["mean"],yerr=stats_m["none"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
#ax1.plot(stats["none"]["centers"],stats["none"]["mean"],'--',color='black',alpha=0.5, dashes=[5, 3])
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Middle panel: best multi-feature RF residuals
ax2.scatter(MASS_TEST, models["obs"]["resid_m"], alpha=.8, s=6, color='C1')
ax2.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax2.errorbar(stats_m["obs"]["centers"],stats_m["obs"]["mean"],yerr=stats_m["obs"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Bottom panel: relative scatter
ax3.semilogx(xbins, np.ones(len(xbins)), color='C0') #blue

for k, v in models.items():
    if k == "none":
        continue
    st = stats_m[k]
    ratio = st["std"]/base["std"]
    err = ratio*np.sqrt((st["std_err"]/st["std"])**2
                        + (base["std_err"]/base["std"])**2)
    ax3.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)
ax3.fill_between(base["centers"], 0, NOISE_FLOOR/base["std"], color="0.85", zorder=0)

# ax3.semilogx(xbins, rfResults["MassConc"]["std"]/rfResults["Mass"]["std"], color='C1') #orange
# ax3.semilogx(xbins, rfResults["ObsNoSlope"]["std"]/rfResults["Mass"]["std"], color='C2') #green
# ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C3') #red
#ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C4') #purple

# Axis labels — updated for bias prediction
# fig.text(0.66, 0.95, r"$b^{(1)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c}]$", rotation=0, fontsize=16)
# fig.text(0.23, 0.62, r"$b^{(2)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}]$",
#           rotation=0, fontsize=16)
fig.text(0.78, 0.95, models["none"]["label"], rotation=0, fontsize=16)
fig.text(0.62, 0.62, models["obs"]["label"], rotation=0, fontsize=16)

fig.text(-0.02, 0.75, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=17)
fig.text(-0.02, 0.40, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=17)
fig.text(0.01, 0.12, r"rel. scatter", rotation=90, fontsize=18)

# Legend annotations for scatter panel — adjust positions to match your curves
text_locations = [
    (0.64, 0.24), 
    (0.64, 0.02), 
    (0.64, 0.28), 
    (0.64, 0.20), 
    (0.64, 0.16), 
    (0.64, 0.04), 

]
for loc, v, in zip(text_locations, models.values()):
    fig.text(loc[0],loc[1], v["label"], fontsize=15, color=v["color"])

# fig.text(0.64, 0.28, r"RF[$M_\mathrm{200c}$]", fontsize=15, color='C0')
# fig.text(0.64, 0.23, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}$]", fontsize=15, color='C1')
# fig.text(0.65, 0.195, r"RF[$M_\mathrm{200c},M_*/M_\mathrm{gas}$]", fontsize=16, color='C2')
# fig.text(0.54, 0.17, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c$]", fontsize=15, color='C2')
# fig.text(0.48, 0.13, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}$]", fontsize=15, color='C3')
# fig.text(0.40, 0.02, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}$]", fontsize=15, color='C4')

plt.xlabel(r'$M_\mathrm{200c}[h^{-1} M_\odot]$', fontsize=19)
fig.text(0.5, 1, r'$z=0$', fontsize=19)

#plt.savefig(PLOTDIR+'scatter_hse_rf_z0z05.png', bbox_inches='tight', dpi=150)


In [ ]:
# TODO: show that the hydrostatic mass bias is related to the Y concentration parameter using a random forest regressor and symbolic regression

### Feature Importance for $b$

In [ ]:
import shap

expl = shap.TreeExplainer(models["obs"]["rf"], feature_names=feat_obs)
sv = expl(models["obs"]["X"][maskTest]) #expl(Xo[test][:500])
fig, ax = plt.subplots(figsize=(8, 5))
shap.plots.beeswarm(sv, show=False, ax=ax, plot_size=None)
ax.set_title("SHAP: hydrostatic bias")
plt.tight_layout()
plt.savefig(PLOTDIR + "shap_bias.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 14. SR for $M_{HSE}$

In [ ]:
from pysr import PySRRegressor


In [ ]:
CANDIDATE_FEATURES_HSE_OBS

In [ ]:

#maskTest = rng.choice([True, False], size=len(m200_rf), p=[0.5, 0.5]) # Define test set
# Y200S = Y200[maskTest]; m200S=m200[maskTest]

SR_FEATURES = ['c_Y_0p50', 'c_Y_0p15', 'c_gas_0p50', 'c_gas_0p15', 'Y_ratio_0p65',
 'M_gas','M_star',
 'ellip_projected']

Xs, ys, ws = make_xy(SR_FEATURES)

model = PySRRegressor(
    maxsize=15,
    maxdepth=10,
    niterations=10000,  # < Increase me for better results
    timeout_in_seconds=60*60*6,
    binary_operators=["+", "-", "*", "/",'pow'],
    unary_operators=[
        "exp",
        "inv(x) = 1/x",
        "log",
        "square",
        # ^ Custom operator (julia syntax)
    ],
    nested_constraints={
        "exp":    {"exp": 0, "log": 0, "pow": 2},
        "log":    {"exp": 0, "log": 0, "pow": 2},
        "square": {"square": 2, "exp": 0, "log": 0},
        "pow":    {"pow": 2, "exp": 2, "log": 0},
    },
    constraints={
        "pow": (9, 9),      # base up to complexity 9, exponent must be complexity-1 (constant or lone var)
        "exp": 9,           # exp can't wrap huge subtrees
        "log": 9,
        "square": 9,
        "/": (-1, 9),       # numerator unconstrained; denominator bounded
    },
    procs = 12,
    parallelism="multithreading",
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    # ^ Define operator for SymPy as well
    elementwise_loss="loss(x, y, w) = w * (x-y)^2"    # ^ Custom loss function (julia syntax)
)

In [ ]:
model.fit(Xs[~maskTest], ys[~maskTest], weights=ws[~maskTest], variable_names=SR_FEATURES)

In [ ]:
model.equations_

### Pareto Curve

In [ ]:
import matplotlib.ticker as ticker

plt.semilogy(model.equations_['complexity'],
             model.equations_['loss'] * 100)

ax = plt.gca()

plt.xlim([1, 10])
plt.ylim([0.6, 1.6])
plt.xticks([2,4,6,8])

# --- Clean y ticks ---
yticks = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6]
ax.set_yticks(yticks)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))

# --- REMOVE scientific notation offset text ---
ax.yaxis.offsetText.set_visible(False)

# --- x ticks
ax.xaxis.set_minor_locator(ticker.NullLocator())
ax.tick_params(axis='x', which='minor', bottom=False)

# --- Optional: cleaner minor ticks for log scale ---
ax.yaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2.75, 5, 0.25)))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())
ax.tick_params(axis='y', which='minor', left=True)

# --- Grid lines ---
ax.grid(True, which='major', axis='both', linestyle='-', alpha=0.4)
ax.grid(True, which='minor', axis='y', linestyle=':', alpha=0.25)


plt.title('Pareto curve')
plt.xlabel('Complexity')
plt.ylabel(r'Loss ($\times 10^{-2}$)')

plt.savefig(PLOTDIR+'hse_pareto_sr.png', bbox_inches='tight', dpi=150)




In [ ]:
#bResid_SR = model.predict(inp[maskTest])  UNSURE which equation or equations .predict works

### SR Model Fitment

In [ ]:
from scipy.optimize import least_squares

# Fit models for proxies created by Wadekar
def sr_model_eq(cy_0p15, cgas_0p50, A, B):
    """b = c_Y_0p15^A/c_gas_0p50^B"""
    return (cy_0p15**A / cgas_0p50**B)

# Pack features the way scipy expects

X_sr_eq, y_sr_eq, w_sr_eq = make_xy(["c_Y_0p15", "c_gas_0p50"])

p0_sq_eq = np.array([2, 4]) 

def residual_sr_model(theta):
    c_Y_0p15 = X_sr_eq[~maskTest, 0]
    c_gas_0p50 = X_sr_eq[~maskTest, 1]
    pred = sr_model_eq(c_Y_0p15, c_gas_0p50, *theta)
    return np.sqrt(w_sr_eq[~maskTest]) * (pred - y_sr_eq[~maskTest])

sol_sr_eq = least_squares(residual_sr_model, x0=p0_sq_eq, max_nfev=20000)

popt_sr_eq = sol_sr_eq.x   # [A]
A_sr_eq = popt_sr_eq[0]
B_sr_eq = popt_sr_eq[1]

print("Model:")
print(f"  A     = {A_sr_eq:.4e}")
print(f"  B     = {B_sr_eq:.4e}")
print(f"  success, cost = {sol_sr_eq.success}, {sol_sr_eq.cost:.6e}")

In [ ]:
sr_pred = sr_model_eq(X_sr_eq[maskTest, 0],
                     X_sr_eq[maskTest, 1],
                     *popt_sr_eq)

sr_model_entry = dict(
    resid=sr_pred - y_sr_eq[maskTest],
    resid_m=(sr_pred - y_sr_eq[maskTest])/(1-sr_pred),
    long_label=r"$b_{\mathrm{SR}} = c_\mathrm{Y,0.15*R200c}^A / c_\mathrm{gas,0.50*R200c}^B$",
    label=r"$b_{\mathrm{SR}}$",
    color="C6"
)

sr_stats = binned_stats(MASS_TEST, sr_model_entry["resid"], MASS_EDGES)

all_models = {
    **models,
    "sr": sr_model_entry
}

all_stats = {
    **stats,
    "sr": sr_stats
}

### Plot Figures

In [ ]:
fig = plt.figure(num=None, figsize=(5.3, 9.333))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

ax1 = fig.add_axes([0.15, 0.7425, 0.84, 0.2475], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax2 = fig.add_axes([0.15, 0.495,  0.84, 0.2475], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax3 = fig.add_axes([0.15, 0.2475, 0.84, 0.2475], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(-0.22,0.22))
ax4 = fig.add_axes([0.15, 0.0,    0.84, 0.2475], xlim=(1e14/H_LITTLE,2e15/H_LITTLE), ylim=(0.15,1.25))

ax1.set_yscale('linear')
ax1.set_xscale('log')
ax2.set_yscale('linear')
ax2.set_xscale('log')
ax3.set_yscale('linear')
ax3.set_xscale('log')
ax1.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax2.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax3.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax4.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0, 1.2])

ax1.scatter(MASS_TEST, all_models["none"]["resid"], alpha=.8, s=6, color='C1') #ax1.scatter(m200_rf[maskTest]*1e10, bResid_M_only_rf, alpha=.8, s=6, color='C1')
ax1.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax1.errorbar(all_stats["none"]["centers"],all_stats["none"]["mean"],yerr=all_stats["none"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
#ax1.plot(stats["none"]["centers"],stats["none"]["mean"],'--',color='black',alpha=0.5, dashes=[5, 3])
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Top panel: Y-only RF residuals as baseline
ax2.scatter(MASS_TEST, all_models["obs"]["resid"], alpha=.8, s=6, color='C1') #ax1.scatter(m200_rf[maskTest]*1e10, bResid_M_only_rf, alpha=.8, s=6, color='C1')
ax2.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax2.errorbar(all_stats["obs"]["centers"],all_stats["obs"]["mean"],yerr=all_stats["obs"]["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
#ax1.plot(stats["none"]["centers"],stats["none"]["mean"],'--',color='black',alpha=0.5, dashes=[5, 3])
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Middle panel: best multi-feature RF residuals
ax3.scatter(MASS_TEST, all_models['sr']["resid"], alpha=.8, s=6, color='C1')
ax3.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)
ax3.errorbar(all_stats['sr']["centers"],all_stats['sr']["mean"],yerr=all_stats['sr']["std"], fmt='--',color='black',alpha=0.5, dashes=[5, 3], capsize=2)
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.4)

# Bottom panel: relative scatter
ax4.semilogx(xbins, np.ones(len(xbins)), color='C0') #blue

for k, v in all_models.items():
    if k == "none":
        continue
    st = all_stats[k]
    ratio = st["std"]/base["std"]
    err = ratio*np.sqrt((st["std_err"]/st["std"])**2
                        + (base["std_err"]/base["std"])**2)
    ax4.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)

ax4.fill_between(base["centers"], 0, NOISE_FLOOR/base["std"], color="0.85", zorder=0)

# ax3.semilogx(xbins, rfResults["MassConc"]["std"]/rfResults["Mass"]["std"], color='C1') #orange
# ax3.semilogx(xbins, rfResults["ObsNoSlope"]["std"]/rfResults["Mass"]["std"], color='C2') #green
# ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C3') #red
#ax3.semilogx(xbins, rfResults["Obs"]["std"]/rfResults["Mass"]["std"], color='C4') #purple

# Axis labels — updated for bias prediction
# fig.text(0.66, 0.95, r"$b^{(1)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c}]$", rotation=0, fontsize=16)
# fig.text(0.23, 0.62, r"$b^{(2)}_\mathrm{pred}=\mathrm{RF}[M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}]$",
#           rotation=0, fontsize=16)
fig.text(0.78, 0.96, (all_models["none"]["label"]), rotation=0, fontsize=16)
fig.text(0.62, 0.7125, (all_models["obs"]["label"]), rotation=0, fontsize=16)
fig.text(0.40, 0.465, (all_models["sr"]["long_label"]), rotation=0, fontsize=16)

fig.text(-0.02, 0.81, r"$b_\mathrm{pred} - b_\mathrm{true}$", rotation=90, fontsize=18)
fig.text(-0.02, 0.5625, r"$b_\mathrm{pred} - b_\mathrm{true}$", rotation=90, fontsize=18)
fig.text(-0.02, 0.315, r"$b_\mathrm{pred} - b_\mathrm{true}$", rotation=90, fontsize=18)
fig.text(0.01, 0.0675, r"rel. scatter", rotation=90, fontsize=18)


# Legend annotations for scatter panel — adjust positions to match your curves
text_locations = [
    (0.64, 0.18), 
    (0.64, 0.02), 
    (0.64, 0.20), 
    (0.64, 0.16), 
    (0.64, 0.10),
    (0.64, 0.06),
    (0.64, 0.13), 
 
]
for loc, v, in zip(text_locations, all_models.values()):
    fig.text(loc[0],loc[1], v["label"], fontsize=15, color=v["color"])

# fig.text(0.64, 0.28, r"RF[$M_\mathrm{200c}$]", fontsize=15, color='C0')
# fig.text(0.64, 0.23, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}$]", fontsize=15, color='C1')
# fig.text(0.65, 0.195, r"RF[$M_\mathrm{200c},M_*/M_\mathrm{gas}$]", fontsize=16, color='C2')
# fig.text(0.54, 0.17, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c$]", fontsize=15, color='C2')
# fig.text(0.48, 0.13, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}$]", fontsize=15, color='C3')
# fig.text(0.40, 0.02, r"RF[$M_\mathrm{200c},\left.\frac{d\ln P}{d\ln r}\right|_{R_{200}}, \beta, x_c, \mathbf{c_{Y}}, \mathbf{c_\mathrm{gas}}$]", fontsize=15, color='C4')

plt.xlabel(r'$M_\mathrm{200c}[h^{-1} M_\odot]$', fontsize=19)
fig.text(0.5, 1, r'$z=0$', fontsize=19)

plt.savefig(PLOTDIR+'scatter_hse_rf_sr_z0z05.png', bbox_inches='tight', dpi=150)


In [ ]:
# Create figure
fig = plt.figure(num=None, figsize=(7, 7))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

# Scatter plot
plt.scatter(y_sr_eq[maskTest], sr_pred, alpha=.8, s=6, color='C1')

# Add the diagonal line (y = x) for perfect predictions
min_val = -0.5
max_val = 0.5
plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)

# Set limits and ticks to be consistent on both axes
plt.xlim([min_val, max_val])
plt.ylim([min_val, max_val])

# Customize ticks: set major ticks, remove scientific notation
ax = plt.gca()
ax.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=True)

# Format ticks to avoid scientific notation
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))

# Set minor ticks (optional for finer details)
ax.minorticks_on()

# Labels and title
plt.xlabel(r'$b_\mathrm{true}$')
plt.ylabel(all_models['sr']['long_label'])
#plt.legend()

# Save the plot
plt.savefig(PLOTDIR+'scatter_hse_diagsr_z0.png', bbox_inches='tight', dpi=150)

# Show the plot
plt.show()


In [ ]:
# Create figure
fig = plt.figure(num=None, figsize=(7, 7))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

# Scatter plot
plt.scatter(yo[maskTest], all_models["obs"]["resid"] + yo[maskTest], alpha=.8, s=6, color='C1')

# Add the diagonal line (y = x) for perfect predictions
min_val = -0.5
max_val = 0.5
plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1)

# Set limits and ticks to be consistent on both axes
plt.xlim([min_val, max_val])
plt.ylim([min_val, max_val])

# Customize ticks: set major ticks, remove scientific notation
ax = plt.gca()
ax.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=True)

# Format ticks to avoid scientific notation
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))

# Set minor ticks (optional for finer details)
ax.minorticks_on()

# Labels and title
plt.xlabel(r'$b_\mathrm{true}$')
plt.ylabel(all_models["obs"]["label"])
#plt.legend()

# Save the plot
plt.savefig(PLOTDIR+'scatter_hse_diagrf_z0.png', bbox_inches='tight', dpi=150)

# Show the plot
plt.show()

In [ ]:
import matplotlib.ticker as ticker

for key in ("sr", "obs", "obs_thry"):
    fig = plt.figure(num=None, figsize=(6.5, 6.5))
    Mtrue = T["M200c"][USE][maskTest]
    Mpred = (1.0 + all_models[key]["resid_m"]) * Mtrue
    plt.scatter(np.log10(Mtrue), np.log10(Mpred), alpha=.6, s=6, color='C1')
    lo, hi = np.log10(Mtrue).min(), np.log10(Mtrue).max()
    plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
    plt.xlim(lo, hi); plt.ylim(lo, hi)
    ax = plt.gca()
    ax.tick_params(axis="both", direction="in", which='both', right=True, top=True)
    plt.xlabel(r'$\log_{10} M_\mathrm{true}\ [M_\odot]$')
    plt.ylabel(r'$\log_{10} M_\mathrm{pred}\ [M_\odot]$')
    plt.title(all_models[key]["label"])
    plt.savefig(PLOTDIR + f'pred_vs_true_ym_{key}.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 14. Robustness Checks

In [ ]:
# == R200-error sensitivity of every candidate parameter ======================
# Two R200 errors are propagated into each feature and reported side by side:
#   (1) SWEPT  : a hypothetical +/-5-10% error -> amplification S = |df/f| / |dR200/R200|
#   (2) ACTUAL : the real HSE-vs-true offset r_ratio = (1-b)^(1/3) (median below),
#                the actual radius error an observer using M_HSE would make.
# A feature is fragile if S>1 (a small R200 error moves it MORE than the error).
# Aperture/concentration features are read off the true profile at a shifted
# fraction; HSE-model features (slope, kT, rms) are re-fit at the shifted
# NORMALIZED radius + fit window -- they are computed in R200-normalized
# coordinates, so the r200 SCALAR does not carry the error (perturbing it gives
# all zeros); the normalized radius does, exactly as in the slope argument.
r_ratio = (1 - T["bias"][USE]) ** (1 / 3)
_delta_actual = float(np.median(r_ratio)) - 1.0
print(f"R200^HSE / R200^true : median {np.median(r_ratio):.4f}  "
      f"16-84 [{np.percentile(r_ratio,16):.4f}, {np.percentile(r_ratio,84):.4f}]  "
      f"-> actual median offset {100*_delta_actual:+.1f}%\n")

_sen_idx = USE
def _amp(cum, f):   # amplitude read inside fraction f of R200
    return lambda s: aperture_value(cum, f * s)[_sen_idx]
def _con(cum, f):   # concentration Q(<f)/Q(<1); both apertures shift with R200
    return lambda s: (aperture_value(cum, f * s) / aperture_value(cum, 1.0 * s))[_sen_idx]

_aperture_feats = {
    "M_gas":        _amp(cumGas, 1.00),
    "M_star":       _amp(cumStr, 1.00),
    "Mgas500":      _amp(cumGas, 0.65),   # the gas factor inside Y_X_500
    "Y200_HSE":     _amp(cumY,   1.00),
    "Y500_HSE":     _amp(cumY,   0.65),
    "c_gas_0p50":   _con(cumGas, 0.50),
    "c_gas_0p15":   _con(cumGas, 0.15),
    "c_Y_0p50":     _con(cumY,   0.50),
    "c_Y_0p25":     _con(cumY,   0.25),
    "c_Y_0p15":     _con(cumY,   0.15),
    "Y_ratio_0p65": _con(cumY,   0.65),
}

def _swept_S(fn):
    base = fn(1.0)
    return float(np.nanmax([np.nanmedian(np.abs(fn(1.0 + d) / base - 1.0)) / abs(d)
                            for d in (0.05, 0.10, -0.05, -0.10)]))
def _actual_pct(fn):
    base = fn(1.0)
    return float(100 * np.nanmedian(np.abs(fn(1.0 + _delta_actual) / base - 1.0)))

rows = [(k, _swept_S(fn), _actual_pct(fn)) for k, fn in _aperture_feats.items()]

# HSE-model features: re-fit at the shifted normalized radius + fit window.
_hse_map = {"slope_R200": "slope_fit", "slope_R200_0p65": "slope_fit_0p65",
            "kT_R200": "kT_keV", "gnfw_rms_resid": "rms"}
try:
    _hids = T["halo_id"][USE][:200].astype(int)
    _sw = {k: {d: [] for d in (0.05, 0.10)} for k in _hse_map}
    _ac = {k: [] for k in _hse_map}
    for hid in _hids:
        r0 = hse_bias(P_e_all[hid], rho_all[hid], r200[hid], m200[hid], a_scale=a_scale_all[hid])
        if not r0["ok"]:
            continue
        for d in (0.05, 0.10, _delta_actual):
            fr = (FIT_RANGE[0] * (1 + d), FIT_RANGE[1] * (1 + d))
            rd = hse_bias(P_e_all[hid], rho_all[hid], r200[hid], m200[hid],
                          r_eval=1.0 * (1 + d), fit_range=fr, a_scale=a_scale_all[hid])
            if not rd["ok"]:
                continue
            for k, kk in _hse_map.items():
                if np.isfinite(r0[kk]) and r0[kk] != 0:
                    rel = abs(rd[kk] / r0[kk] - 1.0)
                    if d == _delta_actual:
                        _ac[k].append(rel)
                    else:
                        _sw[k][d].append(rel)
    for k in _hse_map:
        Ssw = np.nanmax([np.nanmedian(_sw[k][d]) / d if _sw[k][d] else np.nan for d in (0.05, 0.10)])
        pac = 100 * np.nanmedian(_ac[k]) if _ac[k] else np.nan
        rows.append((k, float(Ssw), float(pac)))
except Exception as _e:
    print(f"[HSE-feature sensitivity skipped: {_e}]")

print(f"{'feature':16} {'swept S':>8} {'actual %':>9}   verdict")
print("-" * 48)
for k, S, pac in sorted(rows, key=lambda r: -(r[1] if np.isfinite(r[1]) else -1.0)):
    verd = "FRAGILE (S>1)" if (np.isfinite(S) and S > 1) else "robust"
    print(f"{k:16} {S:8.2f} {pac:8.1f}%   {verd}")
print("\nSWEPT S>1 => amplifies a generic R200 error.  ACTUAL % = median move at the")
print("real HSE-implied R200 offset. Prefer low-S, low-actual features for the SR set.")


In [ ]:
NOISE_LEVELS = {          # fractional 1-sigma, your judgement of real data
    "M_hse":           0.15,
    "slope_R200":      0.20,   # hardest: needs P(r) resolved to R200c
    "kT_R200":         0.15,
    "c_Y_0p50":        0.10,
    "c_gas_0p50":      0.08,
    "Mstar_over_Mgas": 0.25,
    "gnfw_rms_resid":  0.30,
    "ellip_projected": 0.15,
}

X_clean, y_all2, w_all2 = make_xy(PRUNED)
rf_clean = RandomForestRegressor(**RF_KW)
rf_clean.fit(X_clean[~maskTest], y_all2[~maskTest], sample_weight=w_all2[~maskTest])
mse_clean = np.mean((rf_clean.predict(X_clean[maskTest]) - y_all2[maskTest])**2)

_rng_n = np.random.default_rng(0)
print(f"clean test MSE {mse_clean:.5f}\n")
print(f"{'perturbed feature':<22} {'noise':>7} {'MSE':>10} {'degradation':>13}")
print("-"*56)
for j, c in enumerate(PRUNED):
    sig = NOISE_LEVELS.get(c, 0.15)
    Xn = X_clean.copy()
    Xn[:, j] *= np.exp(_rng_n.normal(0, sig, len(Xn)))
    m = np.mean((rf_clean.predict(Xn[maskTest]) - y_all2[maskTest])**2)
    print(f"{c:<22} {sig:>7.0%} {m:>10.5f} {100*(m/mse_clean-1):>+12.1f}%")

Xn = X_clean * np.exp(_rng_n.normal(0, 1, X_clean.shape)
                      * np.array([NOISE_LEVELS.get(c, .15) for c in PRUNED]))
m_all = np.mean((rf_clean.predict(Xn[maskTest]) - y_all2[maskTest])**2)
print(f"\nALL features perturbed simultaneously: MSE {m_all:.5f} "
      f"({100*(m_all/mse_clean-1):+.1f}%)")
print(f"compare to the no-model baseline: {np.var(y_all2[maskTest]):.5f}")
print("\nA feature whose perturbation alone destroys most of the skill is a")
print("feature the final equation should not depend on heavily.")

In [ ]:

sub = USE[:400]
print(f"{'fit range':<16} {'median b':>10} {'scatter':>9} {'med rms':>9} "
      f"{'|curvature|':>12} {'conv':>6}")
print("-"*70)
for lo, hi in [(0.03,2.50),(0.03,2.00),(0.03,1.50),(0.03,1.20),(0.03,1.00),
                (0.10,2.00),(0.10,1.20),
                (0.15,1.00),(0.15,1.20),
                (0.20,1.00)]:
    bs, rs, cur, nc = [], [], [], 0
    for i in sub:
        f = fit_gnfw(R_EFF, P_e_all[i], fit_range=(lo, hi))
        if not f["converged"]:
            continue
        nc += 1
        r = hse_bias(P_e_all[i], rho_all[i], r200[i], m200[i],
                     fit_range=(lo, hi), a_scale=a_scale_all[i])
        bs.append(r["bias"]); rs.append(f["rms"])
        # systematic curvature: mean residual in the outer third of the range
        msk = (R_EFF >= lo) & (R_EFF <= hi)
        resid = np.log(P_e_all[i][msk]) - gnfw_lnP(np.log(R_EFF[msk]),
                        f["ln_P0"], f["ln_xc"], f["beta"])
        outer = R_EFF[msk] > (lo + 0.67*(hi-lo))
        cur.append(abs(np.mean(resid[outer])) if outer.sum() > 2 else np.nan)
    print(f"[{lo:.2f}, {hi:.2f}]{'':<4} {np.median(bs):>+10.4f} {np.std(bs):>9.4f} "
          f"{np.median(rs):>9.4f} {np.nanmedian(cur):>12.4f} {nc/len(sub):>5.0%}")

print("\nRead the CURVATURE column, not the scatter column. Systematic residual")
print("in the outer part of the fit means the gNFW is being pulled by the core.")
print("Then quote the spread in median b across defensible ranges as a")
print("SYSTEMATIC, not as something to minimise.")

---
## 17. Y–M scatter analysis

This section targets the scatter of the true mass about the self-similar SZ
proxy,

$$y \equiv \frac{M_{200c}}{A\,Y_{200c}^{3/5}},$$

so a perfect proxy gives $y=1$ for every halo. It is built to sit *next to*
the HSE-bias section above, not duplicate it: it reuses the halo table `T`, the
`qual` quality mask, the finite-everywhere index `USE`, and the
`maskTest`/`MASS_TEST` train/test split, so `USE_Y` ends up identical to `USE`
— same halos, same split as the HSE analysis, an apples-to-apples comparison.

**What is *not* reused is the candidate pool.** The Y-M section builds its own
`CANDIDATE_FEATURES_Y` (next cells) from a single classification dict
`OBSERVABILITY_Y`, because the Y-M problem needs a different pool than the HSE
one: it **excludes `Y200_HSE`/`Y500_HSE`** (the amplitude of the very relation
we are correcting — they define `scatter_target`, so they are not predictors of
its residual, and dropping `Y500_HSE` removes the exact identity
`Y500_HSE = Y_ratio_0p65 * Y200_HSE`), and it **adds the X-ray analogues
`Y_X_500`/`Y_X_200`**. Redundancy grouping, the univariate screen, the subset
search, and the bootstrap all run over this same pool, and `PRUNED_Y` — the
<=8-feature set handed to symbolic regression — is chosen BY HAND from that evidence plus the observable catalog
-- the automated screens rank which features carry signal, but the cut to the final set is a deliberate, physically-motivated choice.


### Build the scatter target

In [ ]:
from scipy.optimize import least_squares

# Fit the proxy amplitude A on the TRAIN split only (mirrors how the HSE
# section's "none" baseline uses median(y_tr), not the full sample) so the
# test-set scatter reported below isn't leaking test halos into the baseline
# it's compared against.
# Anchor the amplitude A to the z=0 TRAIN halos only; self-similarity carries it
# to every snapshot via the E(z)^(-2/5) term below (M = A E(z)^{-2/5} Y^0.6).
assert len(maskTest) == N_USE, \
    "maskTest is stale (len != N_USE) -- re-run the train/test split cell (sec. 12) first."

# Anchor A on the z=0 TRAIN halos when z=0 is in the pool: E(z)=1 there, so A is
# literally the z=0 amplitude and self-similarity carries it to every snapshot via
# the E(z)^{-2/5} term. If z=0 is absent (a single-snapshot run of another epoch),
# fall back to all TRAIN halos, fitting the full E(z)-scaled model so A stays the
# z=0-equivalent amplitude.
_anchor = (~maskTest) & (T["z"][USE] == 0.0)
_anchor_desc = "z=0 train split"
if not _anchor.any():
    _anchor = ~maskTest
    _anchor_desc = "all train halos, E(z)-scaled (no z=0 in pool)"
    print("note: no z=0 halos in the pool -- anchoring A on all train halos via E(z)^{-2/5}")
_Ez_a = T["E_z"][USE][_anchor]
_Y_tr = T["Y200_HSE"][USE][_anchor]
_M_tr = T["M200c"][USE][_anchor]
_w_tr = _M_tr * _Ez_a**(-0.4)                  # match this section's weight convention

_x0_A = float(np.median(_M_tr / (_Ez_a**(-0.4) * _Y_tr**0.6)))   # unit-agnostic initial guess

def _resid_A_ref(theta):
    pred = theta[0] * _Ez_a**(-0.4) * _Y_tr**0.6
    return np.sqrt(_w_tr) * (pred / _M_tr - 1.0)

_sol_A_ref = least_squares(_resid_A_ref, x0=[_x0_A], max_nfev=20000)
A_REF_Y = float(_sol_A_ref.x[0])
print(f"self-similar amplitude ({_anchor_desc}): A = {A_REF_Y:.4e}  "
      f"(M = A * E(z)^-2/5 * Y200_HSE^0.6, Msun)")

_proxy = A_REF_Y * T["E_z"]**(-0.4) * T["Y200_HSE"]**0.6      # M = A E(z)^{-2/5} Y^0.6
T["scatter_target"] = T["M200c"] / _proxy
T["baseline_resid"] = _proxy / T["M200c"] - 1.0

USE_Y = USE     # candidate pool + sample match the HSE section exactly; see intro
N_USE_Y = len(USE_Y)
assert np.all(np.isfinite(T["scatter_target"][USE_Y])) and np.all(T["scatter_target"][USE_Y] > 0), \
    "scatter_target not finite/positive somewhere in USE -- check Y200_HSE / M200c edge cases"

print(f"target y = M200c/(A Y200_HSE^0.6): median {np.median(T['scatter_target'][USE_Y]):.4f}, "
      f"sigma[ln y] {np.std(np.log(T['scatter_target'][USE_Y])):.4f}  "
      f"({N_USE_Y} halos, same sample as USE)")
for _s in SNAPSHOTS:
    _msk = (T["z"][USE_Y] == float(_s.split("=")[1]))
    print(f"   {_s}: median y = {np.median(T['scatter_target'][USE_Y][_msk]):.4f}  "
          f"sigma[ln y] = {np.std(np.log(T['scatter_target'][USE_Y][_msk])):.4f}  (N={_msk.sum()})")

### Export for Agentic 

In [ ]:
import importlib
import mtng_ym.config, mtng_ym.features, export_table
importlib.reload(mtng_ym.config)
importlib.reload(mtng_ym.features)
importlib.reload(export_table)
from export_table import export_table_ym

In [ ]:
import sys
sys.path.insert(0, "/Users/chase/glxy_mass/agentic_ym")   # so `export_table` and `mtng_ym` import
from export_table import export_table_ym

res = export_table_ym(
    T, USE_Y,
    frozen=dict(
        FIT_RANGE=FIT_RANGE, MASS_CUT=MASS_CUT, MAX_LOG_RESID=MAX_LOG_RESID,
        SLOPE_WINDOW=SLOPE_WINDOW, SNAPSHOTS=SNAPSHOTS,
        Y_EXPONENT=0.6, EZ_EXPONENT=-0.4,
        R500_OVER_R200=0.65, CORE_FRAC=0.15,
        ELLIP_LOS_SEED=0, NOISE_FLOOR_LN=0.05,
    ),
    path="/Users/chase/glxy_mass/agentic_ym/halo_table_ym.npz",
    manifest="/Users/chase/glxy_mass/agentic_ym/FROZEN.md",
)

### Full-candidate-pool feature importance (diagnostic)

The cell below first assembles the Y-M candidate pool `CANDIDATE_FEATURES_Y`
(see the intro), then trains one RF on **every** feature in it against
`scatter_target` and reads its SHAP ranking. This is the direct
feature-importance check for the full pool — nothing downstream depends on this
diagnostic; it exists purely to inform, alongside the univariate/subset
diagnostics, how you set `PRUNED_Y` by hand. (SHAP splits credit among correlated
features, so read it as a diagnostic, not the selector — the redundancy
grouping does the pruning.)


In [ ]:
# == Unified Y-M candidate pool ===============================================
# Single source of truth for what may enter the Y-M feature search. TIER = how
# hard to measure; PREMISE = PHYSICAL (a real property) vs STRUCTURAL (defined
# by assuming something known false -- M_hse and anything built from it) vs None
# (THEORETICAL: not obtainable from one real observation). The Y-M pool EXCLUDES
# Y200_HSE / Y500_HSE: those are the AMPLITUDE of the relation we are correcting
# (they define scatter_target), not predictors of its residual -- and dropping
# Y500_HSE also removes the exact identity Y500_HSE = Y_ratio_0p65 * Y200_HSE.
OBSERVABILITY_Y = {
    "M_hse":            dict(tier="ROUTINE-DERIVED",   premise="STRUCTURAL"),
    "slope_R200":       dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "slope_R200_0p65":  dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "slope_local_R200": dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "kT_R200":          dict(tier="DEMONSTRATED-HARD", premise="IDEALIZED-STAND-IN"),
    "gnfw_rms_resid":   dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "beta":             dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "xc":               dict(tier="DEMONSTRATED-HARD", premise="NON-PHYSICAL"),
    "c_Y_0p50":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "c_Y_0p15":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "logit_c_Y_0p50":   dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "c_gas_0p50":       dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "c_gas_0p15":       dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "Y_ratio_0p65":     dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "Y_X_500":          dict(tier="ROUTINE-DERIVED",   premise="NON-PHYSICAL-IDEALIZED-STAND-IN"), # No core excised!! Also uses fake R500c, so not included.
    "Y_X_200":          dict(tier="DEMONSTRATED-HARD", premise="IDEALIZED-STAND-IN"), # Non core excised!!
    "Y200_HSE":         dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "Y500_HSE":         dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "M_gas":            dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "M_star":           dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "Mstar_over_Mgas":  dict(tier="ROUTINE-DERIVED",   premise="PHYSICAL"),
    "ellip_projected":  dict(tier="DEMONSTRATED-HARD", premise="PHYSICAL"),
    "f_gas_hse":        dict(tier="ROUTINE-DERIVED",   premise="STRUCTURAL"),
    "M200c":            dict(tier="SIMULATION",       premise=None),
    "R200c":            dict(tier="SIMULATION",       premise=None),
    "f_gas_true":       dict(tier="SIMULATION",       premise=None),
    "T_over_Tvir":      dict(tier="SIMULATION",       premise=None),
    "ellip_3d":         dict(tier="SIMULATION",       premise=None),
    "triaxiality":      dict(tier="SIMULATION",       premise=None),
    "ellip_proj_mean":  dict(tier="SIMULATION",       premise=None),
}
REDUNDANT_Y = {"beta": "slope_R200", "xc": "slope_R200", "logit_c_Y_0p50": "c_Y_0p50"}   # exact-duplicate shape params
Y_M_BASE_VARS = {"Y200_HSE", "Y500_HSE"}                   # amplitude of the relation, not predictors

CANDIDATE_FEATURES_Y = [
    k for k, v in OBSERVABILITY_Y.items()
    if v["premise"] == "PHYSICAL"
    and v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_Y
    and k not in Y_M_BASE_VARS
    and k in T
]

CANDIDATE_FEATURES_Y_OBS_THEORY = [
    k for k, v in OBSERVABILITY_Y.items()
    if v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_Y
    and k not in Y_M_BASE_VARS
    and k in T
]

CANDIDATE_FEATURES_Y_THEORY = [
    k for k, v in OBSERVABILITY_Y.items()
    if v["premise"] != "PHYSICAL"
    and v["tier"] in ("ROUTINE-DERIVED", "DEMONSTRATED-HARD")
    and k not in REDUNDANT_Y
    and k not in Y_M_BASE_VARS
    and k in T
]

print(f"Y-M candidate pool OBS: {len(CANDIDATE_FEATURES_Y)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_Y))

print(f"Y-M candidate pool OBS+THEORY: {len(CANDIDATE_FEATURES_Y_OBS_THEORY)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_Y_OBS_THEORY))

print(f"Y-M candidate pool THEORY: {len(CANDIDATE_FEATURES_Y_THEORY)} features")
print("   " + ", ".join(CANDIDATE_FEATURES_Y_THEORY))

print(f"excluded as base variables of the relation: {sorted(Y_M_BASE_VARS & set(OBSERVABILITY_Y))}")
print(f"excluded as exact-duplicate shape params:   {sorted(REDUNDANT_Y)}")


In [ ]:
feature_name_map = {
    'slope_R200': r'$\frac{d}{dr}\ln(P(r))|_{\mathrm{R}200c}$',
    'slope_R200_0p65': r'$\frac{d}{dr}\ln(P(r))|_{0.65*\mathrm{R}200c}$',
    'slope_local_R200': r'$\frac{d}{dr}\ln(P(r))|_{\mathrm{R}200c\mathrm{,local}}$',
    'c_Y_0p50': r'$c_{Y,0.50}$',
    'c_Y_0p15': r'$c_{Y,0.15}$',
    'c_gas_0p50': r'$c_{\mathrm{gas},0.50}$',
    'c_gas_0p15': r'$c_{\mathrm{gas},0.15}$',
    'Y_ratio_0p65': r'$c_{Y,0.65}$',
    'M_gas': r'$M_\mathrm{gas}$',
    'M_star': r'$M_*$',
    'Mstar_over_Mgas': r'$M_*/M_\mathrm{gas}$',
    'ellip_projected': r'$e_\mathrm{projected}$'
}

# Translate CANDIDATE_FEATURES_Y into formalized LaTeX names
formalized_feature_names = [feature_name_map.get(c, c) for c in CANDIDATE_FEATURES_Y]

In [ ]:
import shap

RF_KW_Y = dict(n_estimators=300, min_samples_leaf=5, max_depth=None,
               random_state=0, n_jobs=-1)

X_full_Y = np.column_stack([np.asarray(T[c][USE_Y], float) for c in CANDIDATE_FEATURES_Y])
y_full_Y = np.asarray(T["scatter_target"][USE_Y], float)
w_full_Y = np.asarray(T["M200c"][USE_Y] * T["E_z"][USE_Y]**(-0.4), float)

rf_full_Y = RandomForestRegressor(**RF_KW_Y)
rf_full_Y.fit(X_full_Y[~maskTest], y_full_Y[~maskTest], sample_weight=w_full_Y[~maskTest])
mse_full_Y = np.mean((rf_full_Y.predict(X_full_Y[maskTest]) - y_full_Y[maskTest])**2)
print(f"full-pool RF ({len(CANDIDATE_FEATURES_Y)} features) test MSE: {mse_full_Y:.5e}   "
      f"frac var explained: {1 - mse_full_Y/np.var(y_full_Y[maskTest]):.3f}")






expl_full_Y = shap.TreeExplainer(rf_full_Y, feature_names=formalized_feature_names)
sv_full_Y = expl_full_Y(X_full_Y[maskTest])
fig, ax = plt.subplots(figsize=(8, 4))
shap.plots.beeswarm(sv_full_Y, show=False, ax=ax, plot_size=None, max_display=6)
ax.set_title("SHAP: Y-M scatter target, full candidate pool")
plt.tight_layout()
plt.savefig(PLOTDIR + "shap_ym_full_pool.png", dpi=150, bbox_inches="tight")
plt.show()

### Feature selection

`cv_score` and `_est` are reused from the HSE section (a deliberately weak
learner that measures how much signal a feature SET carries). The Spearman
redundancy grouping, however, is **recomputed here** on the Y-M candidate pool
(`groups_Y`, next cell): the pool differs from the HSE one — it drops the base
variables `Y200_HSE`/`Y500_HSE` and adds `Y_X_500`/`Y_X_200` — so the HSE
`groups` would not describe it. `cv_score` is now weighted consistently (both
the fit and the scored MSE use the mass weights), so `frac_var` reflects the
high-mass end the weighting emphasizes.


In [ ]:
# Redundancy on the Y-M pool -- a DIAGNOSTIC only. It reveals which features are
# correlated families so the MANUAL pruning below can choose ONE member per
# family on physical grounds (see the observable catalog). It does NOT auto-pick
# a representative. Threshold 0.80 per the methodology review.
THRESH_Y = 0.90
A_Y = np.column_stack([T[c][USE_Y] for c in CANDIDATE_FEATURES_Y_OBS_THEORY])
rho_Y = np.atleast_2d(spearmanr(A_Y).statistic)

unassigned, groups_Y = set(range(len(CANDIDATE_FEATURES_Y_OBS_THEORY))), []
while unassigned:
    i = min(unassigned)
    g = {i} | {j for j in unassigned if j != i and abs(rho_Y[i, j]) >= THRESH_Y}
    groups_Y.append(sorted(g)); unassigned -= g

print(f"redundancy groups at |spearman| >= {THRESH_Y} (Y-M pool) -- DIAGNOSTIC:")
for g in groups_Y:
    nm = [CANDIDATE_FEATURES_Y_OBS_THEORY[i] for i in g]
    print("   " + (" ~ ".join(nm) if len(nm) > 1 else nm[0]))
print("\nWithin any family, pick ONE member for PRUNED_Y by physics/observability")
print("(e.g. c_gas over c_Y -- no shared Y200 denominator with the target), not")
print("automatically. This grouping is informational; nothing downstream collapses it.")


In [ ]:
y_all_Y = np.asarray(T["scatter_target"][USE_Y], float)
w_all_Y = np.asarray(T["M200c"][USE_Y] * T["E_z"][USE_Y]**(-0.4), float)
base_var_Y = wvar(y_all_Y, w_all_Y)      # weighted, matching cv_score's weighted MSE

uni_Y = []
for c in CANDIDATE_FEATURES_Y_OBS_THEORY:
    v = np.asarray(T[c][USE_Y], float)
    m = cv_score(v[:, None], y_all_Y, w_all_Y)
    uni_Y.append((c, m, 1 - m / base_var_Y, float(spearmanr(v, y_all_Y).statistic)))
uni_Y.sort(key=lambda r: r[1])          # best (lowest MSE / highest frac_var) first
frac_var_Y = {c: fv for c, m, fv, sp in uni_Y}

print(f"weighted var(y) = {base_var_Y:.5e}   (the number to beat)\n")
print(f"{'feature':<22} {'CV MSE alone':>13} {'frac var':>10} {'spearman':>10}")
print("-" * 58)
for c, m, fv, sp in uni_Y:
    print(f"{c:<22} {m:>13.5e} {fv:>10.3f} {sp:>+10.3f}")

# The pool the automated evidence (subset search, bootstrap) works over: EVERY
# feature with standalone signal. No family collapse -- correlated members stay
# visible so the manual pruning can choose among them on physical grounds.
POSITIVE_FRAC_VAR_Y = [c for c, m, fv, sp in uni_Y if fv > 0]
print(f"\n{len(POSITIVE_FRAC_VAR_Y)} features with positive frac_var "
      f"(the pool for the screens below and for manual pruning):")
print("   " + ", ".join(POSITIVE_FRAC_VAR_Y))
print("\nBoth the fit and the score are mass-weighted, so this reflects the")
print("high-mass end the weighting emphasizes. frac_var <= 0 = no standalone signal.")


In [ ]:
# Subset search over the positive-frac_var pool -- EVIDENCE, not a selector. No
# family collapse: every correlated member stays in, so you can see which
# combinations explain the Y-M scatter best and choose PRUNED_Y by hand below.
_subset_pool = POSITIVE_FRAC_VAR_Y[:13]      # cap the exhaustive search for runtime
if len(POSITIVE_FRAC_VAR_Y) > len(_subset_pool):
    print(f"(exhaustive subset search capped to the top {len(_subset_pool)} of "
          f"{len(POSITIVE_FRAC_VAR_Y)} positive features by frac_var)\n")

subset_results_Y = {}
_sizes = list(range(1, min(6, len(_subset_pool) + 1)))
for size in _sizes:
    rows = []
    for combo in combinations(_subset_pool, size):
        M = np.column_stack([T[c][USE_Y] for c in combo])
        rows.append((combo, cv_score(M, y_all_Y, w_all_Y)))
    rows.sort(key=lambda r: r[1])
    subset_results_Y[size] = rows
    print(f"--- best subsets of size {size} ---")
    for combo, m in rows[:5]:
        print(f"   {1 - m / base_var_Y:>6.3f} frac var   MSE {m:.5e}   {' + '.join(combo)}")

best_by_size_Y = {s: r[0] for s, r in subset_results_Y.items()}
print("\nPLATEAU DIAGNOSTIC -- where does adding a feature stop paying? (advisory)")
prev = base_var_Y
for s in _sizes:
    m = best_by_size_Y[s][1]
    gain = 100 * (prev - m) / base_var_Y
    print(f"   size {s}: frac var {1 - m / base_var_Y:.3f}   gain over size {s-1}: {gain:+.1f}% of var(y)")
    prev = m
print("\nRead this WITH the observable catalog and the redundancy families above,")
print("then set PRUNED_Y by hand. This cell does not choose it for you.")


In [ ]:
# Bootstrap stability over the positive-frac_var pool -- EVIDENCE, not a selector.
# Resamples strictly inside the training half; reports how often each feature
# lands in the best size-SIZE subset. Informs, does not set, the manual PRUNED_Y.
_boot_pool = POSITIVE_FRAC_VAR_Y[:10]
SIZE_Y = min(5, len(_boot_pool))
N_BOOT_Y = 100
_rng_boot_Y = np.random.default_rng(0)
counts_Y = {c: 0 for c in _boot_pool}
combo_counts_Y = {}
_train_idx = np.where(~maskTest)[0]

for b_i in range(N_BOOT_Y):
    sub = _rng_boot_Y.choice(_train_idx, size=int(0.7 * len(_train_idx)), replace=False)
    best, best_m = None, np.inf
    for combo in combinations(_boot_pool, SIZE_Y):
        M = np.column_stack([T[c][USE_Y][sub] for c in combo])
        m = cv_score(M, y_all_Y[sub], w_all_Y[sub], n_splits=3, seed=b_i)
        if m < best_m:
            best, best_m = combo, m
    for c in best:
        counts_Y[c] += 1
    combo_counts_Y[best] = combo_counts_Y.get(best, 0) + 1

print(f"selection frequency over {N_BOOT_Y} resamples (best subset of size {SIZE_Y}, train only):")
for c, k in sorted(counts_Y.items(), key=lambda kv: -kv[1]):
    print(f"   {c:<22} {k/N_BOOT_Y:>5.0%}  " + "#" * int(30 * k / N_BOOT_Y))
print("\nwinning combinations:")
for combo, k in sorted(combo_counts_Y.items(), key=lambda kv: -kv[1])[:5]:
    print(f"   {k/N_BOOT_Y:>5.0%}  {' + '.join(combo)}")


In [ ]:
# == MANUAL pruned set for symbolic regression ================================
# Chosen BY HAND, not derived. The screens above (frac_var, subset search,
# bootstrap, and the full-pool SHAP) show which features carry Y-M signal; the
# observable catalog (docs/observable_catalog_YM.md) says what each feature is,
# how observable it is, and what motivates it. Cut to <= ~8 physically-motivated
# features here, keeping ONE representative per correlated family (see groups_Y)
# on physical grounds -- e.g. prefer c_gas_0p50 over c_Y_0p50 (its concentration
# does not share the target's Y200 denominator), or the more directly observable
# member. Edit this list; nothing upstream sets it.
PRUNED_Y = [
    "c_gas_0p15",       # gas concentration, at smaller aperature -- Wadekar's leading Y-M-scatter driver, leakage-safe
    "slope_R200_0p65",
    "ellip_projected",
    "f_gas_hse", #"slope_local_R200",
    "c_gas_0p50",       # gas concentration -- Wadekar's leading Y-M-scatter driver, leakage-safe
    "c_Y_0p50",         # Y concentration
    "Mstar_over_Mgas",  # baryon partition -- Wadekar's #2 driver
    "slope_R200",       # outer pressure log-slope -- accretion / thermodynamic state
    # candidates to weigh by hand (see catalog): Y_X_500, Y_X_200, c_gas_0p15,
    # gnfw_rms_resid, ellip_projected, a dynamical-state proxy (not yet available)
]

# soft guards -- warn, never auto-edit the choice
_missing = [c for c in PRUNED_Y if c not in T]
assert not _missing, f"PRUNED_Y has features not in T: {_missing}"
_notpos = [c for c in PRUNED_Y if c not in set(POSITIVE_FRAC_VAR_Y)]
if _notpos:
    print(f"note: PRUNED_Y includes features with frac_var <= 0 alone: {_notpos} "
          f"(fine as a correction term if the catalog justifies it -- just be deliberate)")
if len(PRUNED_Y) > 8:
    print(f"warning: PRUNED_Y has {len(PRUNED_Y)} features; symbolic regression wants <= ~8")
print(f"manual PRUNED_Y ({len(PRUNED_Y)}): {PRUNED_Y}")


### Random forest

### Feature classification: TIER x PREMISE, and three model feature sets

Every candidate carries a `TIER` (how hard is it to actually go measure) and a
`PREMISE` (does it correspond to a real physical quantity, or is it *defined*
by assuming something known false -- only `M_hse`/`f_gas_hse` have this
problem). `REDUNDANT_Y` drops statistically duplicate features (`beta`/`xc`,
degenerate with `slope_R200`) independent of observability.

Three feature sets are built from this classification and run through RF
below, each isolating a different question:
- **concentrations, regardless of observability** -- does the leading
  physical hypothesis (gas/SZ concentration) explain the scatter at all,
  independent of how hard any individual aperture is to get?
- **routine-derived, physical only** -- what's achievable with data any
  modest cluster observation already gives you today (excludes `M_hse`,
  which is STRUCTURAL, not just hard)?
- **all observable, physical** -- the ceiling using every legitimately
  physical feature demonstrated in the literature, routine or not, still
  excluding anything STRUCTURAL or THEORETICAL.

In [ ]:
# OBSERVABILITY_Y / REDUNDANT_Y / Y_M_BASE_VARS are defined in the unified-pool
# cell above. Here we build the three physics-motivated RF COMPARISON batches --
# each isolates a different question. These are NOT the SR input; the SR-bound
# set is PRUNED_Y from the evidence-driven screen above.
CONCENTRATION_FEATURES_Y = [c for c in
    ["c_Y_0p50", "c_Y_0p15",
     "c_gas_0p50", "c_gas_0p15", "Y_ratio_0p65"] if c in T]

CONCENTRATION_FEATURES_Y_MHSE = [c for c in
    ["c_Y_0p50", "c_Y_0p15",
     "c_gas_0p50", "c_gas_0p15", "Y_ratio_0p65", "M_hse"] if c in T]

ROUTINE_PHYSICAL_Y = [k for k, v in OBSERVABILITY_Y.items()
                      if v["tier"] == "ROUTINE-DERIVED" and v["premise"] == "PHYSICAL"
                      and k not in Y_M_BASE_VARS 
                      and k not in REDUNDANT_Y
                      and k in T]

ALL_OBSERVABLE_PHYSICAL_Y = list(CANDIDATE_FEATURES_Y)   # the observable-physical ceiling = the pool

print(f"(a) concentrations (regardless of observability): {len(CONCENTRATION_FEATURES_Y)} -> {CONCENTRATION_FEATURES_Y}")
print(f"(b) routine-derived, physical:                    {len(ROUTINE_PHYSICAL_Y)} -> {ROUTINE_PHYSICAL_Y}")
print(f"(c) all observable, physical (= pool):            {len(ALL_OBSERVABLE_PHYSICAL_Y)} -> {ALL_OBSERVABLE_PHYSICAL_Y}")
print(f"--> SR-bound pruned set (PRUNED_Y):               {len(PRUNED_Y)} -> {PRUNED_Y}")


In [ ]:
feat_Y_concentration = CONCENTRATION_FEATURES_Y      # (a) concentrations, regardless of observability
feat_Y_concentration_mhse = CONCENTRATION_FEATURES_Y_MHSE      # (a) concentrations, regardless of observability
feat_Y_routine        = ROUTINE_PHYSICAL_Y             # (b) routine-derived, physical only
feat_obs_Y            = CANDIDATE_FEATURES_Y           # (c) all observable, physical (routine + demonstrated-hard)
feat_obs_thry_Y       = CANDIDATE_FEATURES_Y_OBS_THEORY           # (d) all observable, physical (routine + demonstrated-hard) AND theoretical

def make_xy_Y(feats):
    '''Aligned to USE_Y for every feature set. Never returns a mismatched length.'''
    X = np.column_stack([np.asarray(T[c][USE_Y], float) for c in feats])
    y = np.asarray(T["scatter_target"][USE_Y], float)
    w = np.asarray(T["M200c"][USE_Y] * T["E_z"][USE_Y]**(-0.4), float)
    return X, y, w


def rf_resid_Y(feats, label, color):
    X, y, w = make_xy_Y(feats)
    rf = RandomForestRegressor(**RF_KW_Y)
    rf.fit(X[~maskTest], y[~maskTest], sample_weight=w[~maskTest])
    resid = rf.predict(X[maskTest]) / y[maskTest] - 1.0   # M_pred/M_true - 1
    return dict(resid=resid, label=label, color=color, rf=rf, X=X, y=y, feats=feats)


def binned_stats_Y(mass, resid, edges, n_boot=500, seed=0, log_scatter=True):
    '''Binned mean + scatter of a residual, with bootstrap errors.

    log_scatter=True reports sigma[ln(1+resid)] (the Y-M convention); the mean
    is always reported on the raw residual so the top panels read as a bias.
    '''
    rng_b = np.random.default_rng(seed)
    nb = len(edges) - 1
    out = {k: np.full(nb, np.nan) for k in ("std", "std_err", "mean", "mean_err")}
    out["count"] = np.zeros(nb, dtype=int)
    for i in range(nb):
        m = (mass >= edges[i]) & (mass < edges[i+1]) & np.isfinite(resid)
        r = resid[m]
        out["count"][i] = len(r)
        if len(r) < 10:
            continue
        s = np.log1p(r) if log_scatter else r
        out["std"][i]  = np.std(s)
        out["mean"][i] = np.mean(r)
        idx = rng_b.integers(0, len(r), size=(n_boot, len(r)))
        bs = r[idx]
        sb = np.log1p(bs) if log_scatter else bs
        out["std_err"][i]  = np.std(np.std(sb, axis=1))
        out["mean_err"][i] = np.std(np.mean(bs, axis=1))
    out["centers"] = np.sqrt(edges[:-1]*edges[1:])
    return out


y_te_Y = np.asarray(T["scatter_target"][USE_Y], float)[maskTest]

models_Y = {}
# BASELINE: the self-similar proxy itself. Its residual scatter IS the Y-M
# scatter, so every ratio below reads as "fraction of the Y-M scatter left".
models_Y["none"]    = dict(resid=(1.0/y_te_Y) - 1.0, label=r"$Y^{3/5}$", color="C0")
models_Y["conc"]    = rf_resid_Y(feat_Y_concentration, "RF[all concentrations]",      "C2")
# models_Y["conc_mhse"]    = rf_resid_Y(feat_Y_concentration_mhse, "RF[all concentrations + M_hse]",      "C7")
models_Y["routine"] = rf_resid_Y(feat_Y_routine,       "RF[routine-derived only]",    "C3")
models_Y["obs"]     = rf_resid_Y(feat_obs_Y,           "RF[all observable physical]", "C4")
models_Y["obs_thry"]     = rf_resid_Y(feat_obs_thry_Y, "RF[all observable physical + theoretical]", "C5")
models_Y["pruned"]  = rf_resid_Y(PRUNED_Y,             "RF[SR selection]",            "C6")

MASS_EDGES_Y = MASS_EDGES     # reuse the HSE section's mass bins (same USE_Y, same MASS_TEST)
xbins_Y = xbins

stats_Y = {k: binned_stats_Y(MASS_TEST, v["resid"], MASS_EDGES_Y) for k, v in models_Y.items()}
base_Y = stats_Y["none"]
NOISE_FLOOR_Y = 0.05   # from the noise-injection cell in Robustness below (suggested ~0.052)

print("per-model log-scatter in the lowest and highest mass bin:")
for k, st in stats_Y.items():
    print(f"   {models_Y[k]['label']:<32} "
          f"bin0 {st['std'][0]:.4f}+/-{st['std_err'][0]:.4f}   "
          f"binN {st['std'][-1]:.4f}+/-{st['std_err'][-1]:.4f}")

### Plot RF models

In [ ]:
fig = plt.figure(num=None, figsize=(5.3, 7))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

ax1 = fig.add_axes([0.15, 0.66, 0.84, 0.33], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax2 = fig.add_axes([0.15, 0.33, 0.84, 0.33], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax3 = fig.add_axes([0.15, 0.0,  0.84, 0.33], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(0.40, 1.1))

for a in (ax1, ax2):
    a.set_yscale('linear'); a.set_xscale('log')
    a.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax3.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

ax1.scatter(MASS_TEST, models_Y["none"]["resid"], alpha=.8, s=6, color='C1')
ax1.errorbar(base_Y["centers"], base_Y["mean"], yerr=base_Y["std"],
             fmt='--', color='black', alpha=0.5, dashes=[5, 3], capsize=2)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.4)
ax1.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)

mid_Y = models_Y["obs"]; mst_Y = stats_Y["obs"]
ax2.scatter(MASS_TEST, mid_Y["resid"], alpha=.8, s=6, color='C1')
ax2.errorbar(mst_Y["centers"], mst_Y["mean"], yerr=mst_Y["std"],
             fmt='--', color='black', alpha=0.5, dashes=[5, 3], capsize=2)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.4)
ax2.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)

ax3.semilogx(xbins_Y, np.ones(len(xbins_Y)), color='C0')
for k, v in models_Y.items():
    if k == "none":
        continue
    st = stats_Y[k]
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = st["std"]/base_Y["std"]
        err = ratio*np.sqrt((st["std_err"]/st["std"])**2 + (base_Y["std_err"]/base_Y["std"])**2)
    ax3.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)
#ax3.fill_between(base_Y["centers"], 0, NOISE_FLOOR_Y/base_Y["std"], color="0.85", zorder=0)

fig.text(0.78, 0.95, models_Y["none"]["label"], rotation=0, fontsize=16)
fig.text(0.40, 0.62, mid_Y["label"], rotation=0, fontsize=15)
fig.text(-0.02, 0.75, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=17)
fig.text(-0.02, 0.40, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=17)
fig.text(0.01, 0.12, r"rel. scatter", rotation=90, fontsize=17)

# _y0 = 0.30
# for i, (k, v) in enumerate([kv for kv in models_Y.items() if kv[0] != "none"]):
#     fig.text(0.63, _y0 - 0.036*i, v["label"], fontsize=12, color=v["color"])



text_locations = [
    (0.64, 0.27), 
    (0.64, 0.15), 
    (0.64, 0.06), 
    (0.64, 0.12),
    (0.64, 0.03), 
    (0.64, 0.09), 

]
for loc, v, in zip(text_locations, models_Y.values()):
    fig.text(loc[0],loc[1], v["label"], fontsize=15, color=v["color"])



plt.xlabel(r'$M_\mathrm{200c}\ \times E(z)^{2/5}~[M_\odot / h]$', fontsize=18)
fig.text(0.35, 1.0, r'$z=0 + z=0.5$', fontsize=18)
plt.savefig(PLOTDIR + 'scatter_ym_rf.png', bbox_inches='tight', dpi=150)
plt.show()

### Feature importance for the pruned model

In [ ]:
CANDIDATE_FEATURES_Y_OBS_THEORY

In [ ]:
feature_name_map_obs_theory = {
    'slope_R200': r'$\frac{d}{dr}\ln(P(r))|_{\mathrm{R}200c}$',
    'slope_R200_0p65': r'$\frac{d}{dr}\ln(P(r))|_{0.65*\mathrm{R}200c}$',
    'slope_local_R200': r'$\frac{d}{dr}\ln(P(r))|_{\mathrm{R}200c\mathrm{,local}}$',
    'kT_R200': r'$T_\mathrm{gas,R200}$',
    'gnfw_rms_resid': r'$\sigma_\mathrm{gNFW}$',
    'c_Y_0p50': r'$c_{Y,0.50}$',
    'c_Y_0p15': r'$c_{Y,0.15}$',
    'c_gas_0p50': r'$c_{\mathrm{gas},0.50}$',
    'c_gas_0p15': r'$c_{\mathrm{gas},0.15}$',
    'Y_ratio_0p65': r'$c_{Y,0.65}$',
    'Y_X_500': r'$Y_{X,\mathrm{R}500}$',
    'Y_X_200': r'$Y_{X,\mathrm{R}200}$',
    'M_gas': r'$M_\mathrm{gas}$',
    'M_star': r'$M_*$',
    'Mstar_over_Mgas': r'$M_*/M_\mathrm{gas}$',
    'ellip_projected': r'$e_\mathrm{projected}$',
    'f_gas_hse': r'$M_\mathrm{gas}/M_\mathrm{HSE}$',
}

# Translate CANDIDATE_FEATURES_Y into formalized LaTeX names
formalized_feature_names_obs_theory = [feature_name_map_obs_theory.get(c, c) for c in CANDIDATE_FEATURES_Y_OBS_THEORY]

In [ ]:
expl_Y = shap.TreeExplainer(models_Y["obs_thry"]["rf"], feature_names=formalized_feature_names_obs_theory)
sv_Y = expl_Y(models_Y["obs_thry"]["X"][maskTest])
fig, ax = plt.subplots(figsize=(8, 4))
shap.plots.beeswarm(sv_Y, show=False, ax=ax, plot_size=None, max_display=6)
ax.set_title("SHAP: Y-M scatter, RF[obs+thry]")
plt.tight_layout()
plt.savefig(PLOTDIR + "shap_ym_obs_theory.png", dpi=150, bbox_inches="tight")
plt.show()


### Symbolic regression

Same PySR setup style as the HSE section, pointed at `scatter_target`; kept
commented out by default since a real run takes hours. The classic
self-similar-plus-concentration form is refit directly with
`scipy.least_squares` below so this section produces a result without running
PySR.

In [ ]:
from pysr import PySRRegressor

SR_FEATURES_Y = PRUNED_Y
Xs_Y, ys_Y, ws_Y = make_xy_Y(SR_FEATURES_Y)

model_Y = PySRRegressor(
    maxsize=15,
    maxdepth=10,
    niterations=10000,
    timeout_in_seconds=60*60*6,
    binary_operators=["+", "-", "*", "/", "pow"],
    unary_operators=["exp", "inv(x) = 1/x", "log", "square"],
    nested_constraints={
        "exp":    {"exp": 0, "log": 0, "pow": 2},
        "log":    {"exp": 0, "log": 0, "pow": 2},
        "square": {"square": 2, "exp": 0, "log": 0},
        "pow":    {"pow": 2, "exp": 2, "log": 0},
    },
    constraints={"pow": (9, 9), "exp": 9, "log": 9, "square": 9, "/": (-1, 9)},
    procs=12,
    parallelism="multithreading",
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    elementwise_loss="loss(x, y, w) = w * (x - y)^2",
)
model_Y.fit(Xs_Y[~maskTest], ys_Y[~maskTest], weights=ws_Y[~maskTest], variable_names=SR_FEATURES_Y)
model_Y.equations_

### SR model fitment

In [ ]:
def sr_model_eq_Y(Y, cX, Ez, A, alpha, b):
    '''M_pred [Msun] = A * E(z)^(-2/5) * Y^alpha * (1 - b * cX).

    The E(z)^(-2/5) self-similar evolution factor matches the baseline proxy
    (M = A E(z)^-2/5 Y^0.6) and the RF target (scatter_target divides it out).
    Without it, the pooled z=0 + z=0.5 fit carries a ~10% redshift offset that
    inflates the proxy scatter, worst at low mass.'''
    return A * Ez**(-0.4) * Y**alpha * (1.0 - b * cX)

SR_EQ_FEATURES_Y = ["Y200_HSE", "c_gas_0p50"]
X_sr_eq_Y, _, w_sr_eq_Y = make_xy_Y(SR_EQ_FEATURES_Y)
M_true_Y = T["M200c"][USE_Y]
Ez_sr_Y  = T["E_z"][USE_Y]                      # per-halo E(z), aligned to USE_Y

p0_Y = np.array([A_REF_Y, 0.60, 0.6])

def _resid_sr_Y(theta):
    Yv  = X_sr_eq_Y[~maskTest, 0]
    cXv = X_sr_eq_Y[~maskTest, 1]
    pred = sr_model_eq_Y(Yv, cXv, Ez_sr_Y[~maskTest], *theta)
    return np.sqrt(w_sr_eq_Y[~maskTest]) * (pred / M_true_Y[~maskTest] - 1.0)

sol_sr_Y = least_squares(_resid_sr_Y, x0=p0_Y, max_nfev=20000)
A_sr_Y, alpha_sr_Y, b_sr_Y = sol_sr_Y.x
print("SR equation refit:  M_pred = A * E(z)^-2/5 * Y^alpha * (1 - b c_gas)")
print(f"  A     = {A_sr_Y:.4e} Msun")
print(f"  alpha = {alpha_sr_Y:.4f}")
print(f"  b     = {b_sr_Y:.4f}")
print(f"  success, cost = {sol_sr_Y.success}, {sol_sr_Y.cost:.6e}")

In [ ]:
_pred_test_Y = sr_model_eq_Y(X_sr_eq_Y[maskTest, 0], X_sr_eq_Y[maskTest, 1],
                              Ez_sr_Y[maskTest], A_sr_Y, alpha_sr_Y, b_sr_Y)
sr_entry_Y = dict(
    resid=_pred_test_Y / M_true_Y[maskTest] - 1.0,
    label=r"$A\,Y^{\alpha}(1 - b\,c_\mathrm{gas})$",
    color="C7",
)
sr_stats_Y = binned_stats_Y(MASS_TEST, sr_entry_Y["resid"], MASS_EDGES_Y)

all_models_Y = {**models_Y, "sr": sr_entry_Y}
all_stats_Y  = {**stats_Y,  "sr": sr_stats_Y}
print("added SR model. all_models_Y keys:", list(all_models_Y))

### Combined RF + SR figure

In [ ]:
fig = plt.figure(num=None, figsize=(5.3, 9.333))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

ax1 = fig.add_axes([0.15, 0.7425, 0.84, 0.2475], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax2 = fig.add_axes([0.15, 0.495,  0.84, 0.2475], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax3 = fig.add_axes([0.15, 0.2475, 0.84, 0.2475], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax4 = fig.add_axes([0.15, 0.0,    0.84, 0.2475], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(0.40, 1.10))

for a in (ax1, ax2, ax3):
    a.set_yscale('linear'); a.set_xscale('log')
    a.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax4.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

def _panel_Y(ax, key):
    m, st = all_models_Y[key], all_stats_Y[key]
    ax.scatter(MASS_TEST, m["resid"], alpha=.8, s=6, color='C1')
    ax.errorbar(st["centers"], st["mean"], yerr=st["std"], fmt='--', color='black',
                alpha=0.5, dashes=[5, 3], capsize=2)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.4)
    ax.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)

_panel_Y(ax1, "none")
_panel_Y(ax2, "obs")
_panel_Y(ax3, "sr")

ax4.semilogx(xbins_Y, np.ones(len(xbins_Y)), color='C0')
for k, v in all_models_Y.items():
    if k == "none":
        continue
    st = all_stats_Y[k]
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = st["std"]/base_Y["std"]
        err = ratio*np.sqrt((st["std_err"]/st["std"])**2 + (base_Y["std_err"]/base_Y["std"])**2)
    ax4.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)
ax4.fill_between(base_Y["centers"], 0, NOISE_FLOOR_Y/base_Y["std"], color="0.85", zorder=0)

fig.text(0.78, 0.96,  all_models_Y["none"]["label"], rotation=0, fontsize=16)
fig.text(0.55, 0.7125, all_models_Y["obs"]["label"], rotation=0, fontsize=14)
fig.text(0.46, 0.465, all_models_Y["sr"]["label"], rotation=0, fontsize=14)
fig.text(-0.02, 0.81,   r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)
fig.text(-0.02, 0.5625, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)
fig.text(-0.02, 0.315,  r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)
fig.text(0.01, 0.0675,  r"rel. scatter", rotation=90, fontsize=16)

text_locations = [
    (0.64, 0.21), 
    (0.64, 0.15), 
    (0.64, 0.06), 
    (0.64, 0.12),
    (0.64, 0.03), 
    (0.64, 0.09), 
    (0.64, 0.18), 


]
for loc, v, in zip(text_locations, all_models_Y.values()):
    fig.text(loc[0],loc[1], v["label"], fontsize=15, color=v["color"])


plt.xlabel(r'$M_\mathrm{200c}\ \times E(z)^{2/5}\ [M_\odot / h]$', fontsize=18)
fig.text(0.35, 1.0, r'$z=0 + z=0.5$', fontsize=18)
plt.savefig(PLOTDIR + 'scatter_ym_rf_sr.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plt.figure(num=None, figsize=(11.5, 5.5))
fig.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

# Same subplot size (4.45" x 2.31") as before, but with a wider gap between columns
ax1 = fig.add_axes([0.069, 0.55, 0.387, 0.42], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax2 = fig.add_axes([0.604, 0.55, 0.387, 0.42], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax3 = fig.add_axes([0.069, 0.11, 0.387, 0.42], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(-0.22, 0.22))
ax4 = fig.add_axes([0.604, 0.11, 0.387, 0.42], xlim=(1e14/H_LITTLE, 2e15/H_LITTLE), ylim=(0.40, 1.10))

for a in (ax1, ax2, ax3):
    a.set_yscale('linear'); a.set_xscale('log')
    a.set_yticks([-0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2])
ax4.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])

def _panel_Y(ax, key):
    m, st = all_models_Y[key], all_stats_Y[key]
    ax.scatter(MASS_TEST, m["resid"], alpha=.8, s=6, color='C1')
    ax.errorbar(st["centers"], st["mean"], yerr=st["std"], fmt='--', color='black',
                alpha=0.5, dashes=[5, 3], capsize=2)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.4)
    ax.tick_params(axis="both", direction="in", which='both', right=True, top=True, labelbottom=0)

_panel_Y(ax1, "none")
_panel_Y(ax2, "obs")
_panel_Y(ax3, "sr")
ax3.tick_params(labelbottom=True)  # ax3 is now on the bottom row, needs x tick labels

ax4.semilogx(xbins_Y, np.ones(len(xbins_Y)), color='C0')
for k, v in all_models_Y.items():
    if k == "none":
        continue
    st = all_stats_Y[k]
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = st["std"]/base_Y["std"]
        err = ratio*np.sqrt((st["std_err"]/st["std"])**2 + (base_Y["std_err"]/base_Y["std"])**2)
    ax4.errorbar(st["centers"], ratio, yerr=err, fmt="o-", color=v["color"],
                 label=v["label"], ms=4, lw=1.4, capsize=2)
ax4.fill_between(base_Y["centers"], 0, NOISE_FLOOR_Y/base_Y["std"], color="0.85", zorder=0)

# Model labels inside subplots (same relative in-panel positions as before)

fig.text(0.359, 0.919, all_models_Y["none"]["label"], rotation=0, fontsize=16)  # ax1
fig.text(0.632, 0.919, all_models_Y["obs"]["label"],  rotation=0, fontsize=14)  # ax2
fig.text(0.212, 0.479, all_models_Y["sr"]["label"],   rotation=0, fontsize=14)  # ax3

# Per-subplot y-axis titles
fig.text(-0.01, 0.665, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)  # ax1
fig.text(0.525, 0.665, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)  # ax2
fig.text(-0.01, 0.225, r"$M_\mathrm{pred}/M_\mathrm{true} - 1$", rotation=90, fontsize=16)  # ax3
fig.text(0.544, 0.225, r"rel. scatter",                          rotation=90, fontsize=16)  # ax4

# In-panel legend text for ax4
text_locations = [
    (0.830, 0.466),
    (0.830, 0.364),
    (0.830, 0.212),
    (0.830, 0.314),
    (0.830, 0.161),
    (0.830, 0.263),
    (0.830, 0.415),
]
for loc, v in zip(text_locations, all_models_Y.values()):
    fig.text(loc[0], loc[1], v["label"], fontsize=15, color=v["color"])

# Centered x-axis label and title
fig.text(0.530, 0.02, r'$M_\mathrm{200c}\ \times E(z)^{2/5}\ [M_\odot / h]$',
         ha='center', va='bottom', fontsize=18)
fig.text(0.530, 1.0,  r'$z=0 + z=0.5$', ha='center', fontsize=18)

plt.savefig(PLOTDIR + 'scatter_ym_rf_sr_wide.png', bbox_inches='tight', dpi=150)
plt.show()

### Prediction vs truth

In [ ]:
import matplotlib.ticker as ticker

for key in ("sr", "obs"):
    fig = plt.figure(num=None, figsize=(6.5, 6.5))
    Mtrue = T["M200c"][USE_Y][maskTest]
    Mpred = (1.0 + all_models_Y[key]["resid"]) * Mtrue
    plt.scatter(np.log10(Mtrue), np.log10(Mpred), alpha=.6, s=6, color='C1')
    lo, hi = np.log10(Mtrue).min(), np.log10(Mtrue).max()
    plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
    plt.xlim(lo, hi); plt.ylim(lo, hi)
    ax = plt.gca()
    ax.tick_params(axis="both", direction="in", which='both', right=True, top=True)
    plt.xlabel(r'$\log_{10} M_\mathrm{true}\ [M_\odot]$')
    plt.ylabel(r'$\log_{10} M_\mathrm{pred}\ [M_\odot]$')
    plt.title(all_models_Y[key]["label"])
    plt.savefig(PLOTDIR + f'pred_vs_true_ym_{key}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
fig = plt.figure(num=None, figsize=(6.5, 6.5))
Mtrue = T["M200c"][USE_Y][maskTest]
for key, c in zip(("sr", "obs"), ('C0', 'C1')):
    Mpred = (1.0 + all_models_Y[key]["resid"]) * Mtrue
    plt.scatter(np.log10(Mtrue), np.log10(Mpred), alpha=.4, s=4, color=c, label=all_models_Y[key]["label"])
    lo, hi = np.log10(Mtrue).min(), np.log10(Mtrue).max()
    plt.plot([lo, hi], [lo, hi], 'k--', lw=1)

plt.xlim(lo, hi); plt.ylim(lo, hi)
ax = plt.gca()
ax.tick_params(axis="both", direction="in", which='both', right=True, top=True)
plt.xlabel(r'$\log_{10} M_\mathrm{true}\ [M_\odot]$')
plt.ylabel(r'$\log_{10} M_\mathrm{pred}\ [M_\odot]$')
plt.title("Prediction comparison between RF and SR models")
plt.legend()
# plt.savefig(PLOTDIR + f'pred_vs_true_ym_{key}.png', bbox_inches='tight', dpi=150)
plt.show()

### Robustness checks

The Y-M analogues of the HSE robustness section: aperture sensitivity of the
SR correction, mass-cut sensitivity of the relative scatter (no z-split —
only z=0.0 is loaded), and feature-noise injection to set `NOISE_FLOOR_Y`.

In [ ]:
print(f"{'aperture':<12} {'A [Msun]':>14} {'alpha':>8} {'b':>8} {'test sigma[ln]':>16}")
print("-"*62)
for ap_key in ["c_gas_0p15", "c_gas_0p50"]:
    Xap, _, wap = make_xy_Y(["Y200_HSE", ap_key])
    def _r(theta, Xap=Xap, wap=wap):
        pred = sr_model_eq_Y(Xap[~maskTest, 0], Xap[~maskTest, 1], Ez_sr_Y[~maskTest], *theta)
        return np.sqrt(wap[~maskTest]) * (pred / M_true_Y[~maskTest] - 1.0)
    s = least_squares(_r, x0=[A_REF_Y, 0.60, 0.6], max_nfev=20000)
    A_, al_, b_ = s.x
    pred_te = sr_model_eq_Y(Xap[maskTest, 0], Xap[maskTest, 1], Ez_sr_Y[maskTest], A_, al_, b_)
    resid_te = pred_te / M_true_Y[maskTest] - 1.0
    sig = np.std(np.log1p(resid_te))
    print(f"{ap_key:<12} {A_:>14.4e} {al_:>8.4f} {b_:>8.4f} {sig:>16.4f}")
print("\nSpread in b across apertures is a systematic on the correction, not")
print("something to minimise by cherry-picking the aperture.")

In [ ]:
print("mass-cut sensitivity (median rel. scatter, RF[pruned set] vs baseline):")
for cut in [4e13, 8e13, 1.5e14, 3e14]:
    keep = T["M200c"][USE_Y][maskTest] > cut
    if keep.sum() < 50:
        print(f"  > {cut:.1e} Msun: too few halos"); continue
    st_none = binned_stats_Y(MASS_TEST[keep], models_Y["none"]["resid"][keep], MASS_EDGES_Y)
    st_obs  = binned_stats_Y(MASS_TEST[keep], models_Y["obs"]["resid"][keep],  MASS_EDGES_Y)
    with np.errstate(invalid="ignore"):
        rr = np.nanmedian(st_obs["std"]/st_none["std"])
    print(f"  > {cut:.1e} Msun ({keep.sum():5d} halos): {rr:.3f}")

In [ ]:
NOISE_LEVELS_Y = {          # fractional 1-sigma, your judgement of real data
    "Y200_HSE":         0.10,
    "c_Y_0p50":         0.10,
    "c_Y_0p25":         0.12,
    "c_Y_0p15":         0.15,
    "c_gas_0p50":       0.08,
    "c_gas_0p15":       0.12,
    "Mstar_over_Mgas":  0.25,
    "slope_R200":       0.20,
    "kT_R200":          0.15,
    "gnfw_rms_resid":   0.30,
    "ellip_projected":  0.15,
}

Xn0, yn0, wn0 = make_xy_Y(PRUNED_Y)
rf_clean_Y = RandomForestRegressor(**RF_KW_Y)
rf_clean_Y.fit(Xn0[~maskTest], yn0[~maskTest], sample_weight=wn0[~maskTest])
mse_clean_Y = np.mean((rf_clean_Y.predict(Xn0[maskTest]) - yn0[maskTest])**2)

_rng_n_Y = np.random.default_rng(0)
print(f"clean test MSE {mse_clean_Y:.5e}\n")
print(f"{'perturbed feature':<22} {'noise':>7} {'MSE':>12} {'degradation':>13}")
print("-"*58)
for j, c in enumerate(PRUNED_Y):
    sig = NOISE_LEVELS_Y.get(c, 0.15)
    Xp = Xn0.copy()
    Xp[:, j] *= np.exp(_rng_n_Y.normal(0, sig, len(Xp)))
    m = np.mean((rf_clean_Y.predict(Xp[maskTest]) - yn0[maskTest])**2)
    print(f"{c:<22} {sig:>7.0%} {m:>12.5e} {100*(m/mse_clean_Y-1):>+12.1f}%")

Xp = Xn0 * np.exp(_rng_n_Y.normal(0, 1, Xn0.shape)
                  * np.array([NOISE_LEVELS_Y.get(c, .15) for c in PRUNED_Y]))
m_all = np.mean((rf_clean_Y.predict(Xp[maskTest]) - yn0[maskTest])**2)
print(f"\nALL features perturbed simultaneously: MSE {m_all:.5e} "
      f"({100*(m_all/mse_clean_Y-1):+.1f}%)")
print(f"no-model baseline var(y): {np.var(yn0[maskTest]):.5e}")
_floor_est_Y = np.std(np.log1p(rf_clean_Y.predict(Xp[maskTest])/yn0[maskTest]-1))
print(f"\nsuggested NOISE_FLOOR_Y ~ {_floor_est_Y:.3f} (set it above once you've reviewed this)")

## Generating summary data from simulation snapshots

In [ ]:
# This section is given just for reference
# and was used to generate the input data files.
# One can ignore this section.

### From MTNG group catalogs

In [ ]:
# They are in the same format as TNG group data
# https://www.tng-project.org/data/docs/specifications/
# I can share the data for groups in a separate globus repository

In [ ]:
import illustris_python as il
# https://github.com/illustristng/illustris_python

In [ ]:
basePath = '/data/jayw/CCA/Simulations/MTNG'
snap_ind = 264 # 264 for z=0.0 and 214 for z=0.5

In [ ]:
m200_all = il.groupcat.loadHalos(basePath,snap_ind,fields='Group_M_Crit200')
maskMass = m200_all>1e3;
m500_all = il.groupcat.loadHalos(basePath,snap_ind,fields='Group_M_Crit500')
m500_all = m500_all[maskMass]

### From Leander's files

In [ ]:
# The raw data was calculated using Leander's code in the examples folder of https://github.com/leanderthiele/group_particles/tree/master/examples
# I've already stored the data in the numpy format in the data folder so these are provided just for reference

In [ ]:
dire='/data/jayw/CCA/SZ/MTNG/z=0.0/'
os.chdir(dire)

In [ ]:
_ = np.fromfile(dire + 'Raw/grp_M200c.bin', dtype=float)
np.save('M200c.npy', _)
_ = np.fromfile(dire + 'Raw/grp_Y.bin', dtype=float)
np.save('Y200c.npy', _)
_ = np.fromfile(dire + 'Raw/grp_R200c.bin', dtype=float)
np.save('R200c.npy', _)
_ = np.fromfile(dire + 'Raw/grp_T.bin', dtype=float)
np.save('T500c.npy', _)

In [ ]:
temp2=np.fromfile('Raw/grp_pressure_prof.bin', dtype=np.float64).reshape((-1,128))
np.save('y_profiles.npy',temp2)
temp2=np.fromfile('Raw/grp_ne_prof.bin', dtype=np.float64).reshape((-1,128))
np.save('ne_profiles.npy',temp2)
temp2=np.fromfile('Raw/dens_prof_STARS.bin', dtype=np.float64).reshape((-1,128))
np.save('mStar_profiles.npy',temp2)

In [ ]:
_ = np.fromfile('Raw/Mtens_GAS.bin', dtype=np.float64).reshape((-1,9))
np.save('Mtensor_Gas.npy',_)

In [ ]:
# use the following to get the masses within R200c exactly rather than calculating the cumulative over bins
# a=np.fromfile('/data/jayw/CCA/SZ/MTNG/z=0.0/Raw/CM/CM_GAS.bin', dtype=float)[::4]+\
#   np.fromfile('/data/jayw/CCA/SZ/MTNG/z=0.0/Raw/CM/CM_DM.bin', dtype=float)[::4]+\
#   np.fromfile('/data/jayw/CCA/SZ/MTNG/z=0.0/Raw/CM/CM_STARS.bin', dtype=float)[::4]+\
#   np.fromfile('/data/jayw/CCA/SZ/MTNG/z=0.0/Raw/CM/CM_BH.bin', dtype=float)[::4]